# Complete Workflow

This workflow provides a complete working example to develop an unstructured mesh for an integrated hydrologic model based on HUCs.  It is the default workflow for integrated hydrology simulations for Exasheds Simulation Campaign 2.

It uses the following datasets:

* `NHD Plus` for the watershed boundary and hydrography.
* `NED` for elevation
* `NLCD` for land cover/transpiration/rooting depths
* `GLYHMPS` geology data for structural formations
* `SoilGrids 2017` for depth to bedrock and soil texture information
* `SSURGO` for soil data, where available, in the top 2m.

Given some basic inputs (in the next cell) including a NAME, this workflow creates the following files (noting that some suffixes may be appended to the user-provided NAME in homogeneous cases):

* Mesh file: `{NAME}.exo`, includes all labeled sets
* Forcing: DayMet data -- daily raster of precip, RH, incoming radiation, etc.
  - `{NAME}_DayMet_1980_2024.h5`, the DayMet data on this watershed
  - `{NAME}_DayMet_typical_1980_2024.h5`, a "typical year" of DayMet, smoothed for spinup purposes, then looped 40 years
* Forcing: LAI data -- every 4 days, time series by land cover type of LAI.  Note, the raw inputs to this are not done by NAME, but by an (optional, defaults to NAME) MODIS_NAME variable.  Since WW does not currently download MODIS, one might want to use a file of a different name to provide MODIS data.  The times of this MODIS data are hard-coded too -- this is all a bit wonky and will remain so until we get around to adding a file manager for MODIS data.
  - `{NAME}_MODIS_LAI_smoothed_2002_2024.h5`, the LAI, interpolated and smoothed from the raw MODIS data
  - `{NAME}_MODIS_LAI_typical_1980_2024.h5`, a "typical year" of LAI, smoothed for spinup purposes then looped 40 years
* Input files: ATS xml files
  - `spinup-steadystate-{NAME}.xml` the steady-state solution based on uniform application of mean rainfall rate
  - `spinup-cyclic_steadystate-{NAME}.xml` the cyclic steady state based on typical years
  - `transient-{NAME}.xml` the forward model


In [ ]:
from fractions import Fraction

In [ ]:
meshsize, factor, k_factor = 100, 2, Fraction(1) # we define the mesh configuration using the following equation

$\textbf{The area threshold function } A(d) \textbf{ is defined as:}$

$$
A(d) = 
\begin{cases}
A_0 & \text{if } d \leq d_0, \\
A_0 + \dfrac{A_1 - A_0}{d_1 - d_0}(d - d_0) & \text{if } d_0 < d < d_1, \\
A_1 & \text{if } d \geq d_1,
\end{cases}
$$

$\textbf{where the parameters are calculated as:}$

$$
\begin{align}
d_0 &= 
\begin{cases}
m(f - 1) & \text{if } f \geq 2, \\
m & \text{otherwise},
\end{cases}
\\[10pt]
d_1 &= 
\begin{cases}
3m(f - 1) & \text{if } f \geq 2, \\
3m & \text{otherwise},
\end{cases}
\\[10pt]
A_0 &= m^2/2, \\[10pt]
A_1 &= \dfrac{(m f k_{\text{factor}})^2}{2}
\end{align}
$$

**where** $m$ **is meshsize (m),** $f$ **is mesh factor and** $k_{factor}$ **is constant to decouple far-field coarsening from near-stream refinement.**

In [ ]:
near_distance = meshsize/2
far_distance = meshsize
near_length_scale = meshsize
far_length_scale = meshsize*factor

refine_d0 = meshsize*(factor-1) if factor >= 2 else meshsize
refine_d1 = meshsize*(factor-1)*3 if factor >= 2 else meshsize*3
refine_L0 = meshsize
refine_L1 = meshsize*factor


print(refine_L0, refine_L1, refine_d0, refine_d1)

In [ ]:
# these can be turned on for development work
%load_ext autoreload
%autoreload 2
import numpy as np

In [ ]:
# setting up logging first or else it gets preempted by another package
import watershed_workflow.ui
watershed_workflow.ui.setup_logging(1)

In [ ]:
name = 'SBKRMixed' # name the domain, used in filenames, etc
hucs = ['070900060601', '070900060602', '070900060603']
def get_huc12(hucs):
    huc12_list = []
    for huc in hucs:
        if len(huc) == 12:
            huc12_list.append(huc)
        elif len(huc) == 10:
            for i in range(1,20):
                huc12_list.append(huc+str(i).zfill(2))
        elif len(huc) == 8:
            for i in range(1,20):
                for j in range(1,20):
                    huc12_list.append(huc+str(i).zfill(2)+str(j).zfill(2))
        else:
            print('need huc8, huc10 or huc12')
    return huc12_list
hucs = get_huc12(hucs)
print(hucs[:10])

In [ ]:
# Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed.
huc_level = 12 # if provided, an int setting the level at which to include HUC boundaries

# geometric parameters
simplify_hucs = 20 # length scale to target average edge
simplify_rivers = 20 ## what is this??
stream_outlet_width = 100 # half-width to track a labeled set on which to get discharge
ignore_small_rivers = 2 # ignore rivers which have this or fewer reaches.  likely they are irrigation ditches
                        # or other small features which make things complicated but likely don't add much value
prune_by_area_fraction = 0.001 # ignore reaches whose accumulated catchment area is less than this fraction of the
                              # full domain's area
prune_by_area_fraction_waterbodies = None
num_smoothing_sweeps = 5 # number of times to smooth the DEM prior to elevating

# simulation control
start_year = 1980  # year to start and end simulation simulation -- note these start and end Oct 1 of the year
end_year = 2024
min_porosity = 0.05 # minimum porosity considered too small
max_permeability = 1.e-10 # max value allowed for permeability
max_vg_alpha = 1.e-3 # max value of van Genuchten's alpha -- our correlation is not valid for some soils

# triangle refinement control
include_rivers = True
# refine_d0 = 100
# refine_d1 = 500
# refine_A0 = 8000
# refine_A1 = 50000
# meshsize = 300
# factor = 1
# refine_d0 = meshsize*3
# refine_A0 = meshsize**2/2
# refine_d1 = meshsize*15
# refine_A1 = (np.round(meshsize*factor))**2/2

# soil structure
use_geologic_layer = True

# logistics
generate_plots = True # plots take time to make and aren't always needed
generate_daymet = False # potentially don't do Met data forcing
generate_modis = False

include_heterogeneous = True
include_homogeneous = False # if true, also write files for homogeneous runs
include_homogeneous_wrm = False # if true, also write files for homogeneous WRMs
include_homogeneous_wrm_porosity = False # if true, also write files for homogeneous porosity and WRMs
include_homogeneous_wrm_permeability = False # if true, also write files for homogeneous perm and WRMs

log_to_file = False  # if true, write to file instead of in the notebook output
figsize = (6,6)
figsize_3d = (8,6)

In [ ]:
# parameter checking
assert(ignore_small_rivers == None or (ignore_small_rivers >= 0 and ignore_small_rivers <= 100))
assert(prune_by_area_fraction == None or (prune_by_area_fraction >= 0 and prune_by_area_fraction < 1))

if type(hucs) is str:
    assert(hucs[0] == '[')
    assert(hucs[-1] == ']')
    hucs = hucs[1:-1]
    hucs = hucs.split(',')
    hucs = [h.strip() for h in hucs]
    if hucs[-1] == '':
        hucs = hucs[:-1]

if huc_level is None:
    huc_level = len(hucs[0])
else:
    assert(huc_level >= len(hucs[0]))
huc_key = f'HUC{huc_level}'

if prune_by_area_fraction_waterbodies is None:
    prune_by_area_fraction_waterbodies = prune_by_area_fraction * 0.1

In [ ]:
# a dictionary of outputs -- will include all filenames generated
outputs = {}

In [ ]:
import os,sys
import logging
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm as pcm
import shapely
import pandas as pd
import geopandas as gpd
import cftime, datetime
pd.options.display.max_columns = None
pd.options.display.max_rows = 20
plt.rcParams['figure.dpi'] = 150

import watershed_workflow 
import watershed_workflow.config
import watershed_workflow.sources
import watershed_workflow.utils
import watershed_workflow.plot
import watershed_workflow.mesh
import watershed_workflow.regions
import watershed_workflow.meteorology
import watershed_workflow.land_cover_properties
import watershed_workflow.resampling
import watershed_workflow.condition
import watershed_workflow.io
import watershed_workflow.sources.standard_names as names

# set the default figure size for notebooks
plt.rcParams["figure.figsize"] = (8, 6)


In [ ]:

# ats_input_spec library
import ats_input_spec
import os
ats_input_spec.set_amanzi_source('/Users/niro298/watershed_workflow/ww_v2_mar3_2026/ats_input_spec')
import ats_input_spec.public
import ats_input_spec.io

# amanzi_xml, included in AMANZI_SRC_DIR/tools/amanzi_xml
import amanzi_xml.utils.io as aio
import amanzi_xml.utils.search as asearch
import amanzi_xml.utils.errors as aerrors

In [ ]:
import exodus

In [ ]:
# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs = watershed_workflow.crs.daymet_crs
crs_daymet = crs
crs

In [ ]:
from fractions import Fraction

def convert_k_factor_to_text(k_factor):
    # Check if k_factor is a whole number
    if k_factor.denominator == 1:
        k_factor_text = str(k_factor.numerator)
    else:
        # Express as fraction: Numerator/Denominator
        k_factor_text = f"{k_factor.numerator}by{k_factor.denominator}"
    
    return k_factor_text
k_factor_text = convert_k_factor_to_text(k_factor)
print(k_factor_text)  # Output: '1'


## Sources and setup

Next we set up the source watershed and coordinate system and all data sources for our mesh.  We will use the CRS that is included in the shapefile.

In [ ]:
logging.info("")
logging.info(f"Meshing shape: {hucs}")
logging.info("="*30)

A wide range of data sources are available; here we use the defaults except for using NHD Plus for watershed boundaries and hydrography (the default is NHD, which is lower resolution and therefore smaller download sizes).

In [ ]:
# set up a dictionary of source objects
sources = watershed_workflow.sources.getDefaultSources()
sources['hydrography'] = watershed_workflow.sources.hydrography_sources['NHDPlus HR']
sources['HUC'] = watershed_workflow.sources.huc_sources['WBD']
sources['DEM'] = watershed_workflow.sources.dem_sources['3DEP 10m']
sources['geologic structure']=watershed_workflow.sources.ManagerGLHYMPS('data/clipped_input/GLHYMPS_SBKRMixed_buffer5km.shp')
sources['depth to bedrock']=watershed_workflow.sources.ManagerRaster('data/clipped_input/BDTICM_M_250m_SBKRMixed_buffer5km.tif')
watershed_workflow.sources.logSources(sources)

In [ ]:
my_hucs = []
sources['HUC'].setLevel(huc_level)
for huc in hucs:
  ws = sources['HUC'].getShapesByID([huc], out_crs=crs)
  my_hucs.append(ws)

import geopandas as gpd
my_hucs = gpd.GeoDataFrame(pd.concat(my_hucs,
ignore_index=True), crs=crs)
watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)
watershed.plot()


In [ ]:
# create necessary folder paths
try:
    os.makedirs(f'../data-processed/{name}')
except FileExistsError:
    pass
    
try:
    os.makedirs(f'./images/{name}')
except FileExistsError:
    pass

try:
    os.makedirs(f'./shapes')
except FileExistsError:
    pass

try:
    os.makedirs(f'./storecsv/{name}')
except FileExistsError:
    pass

try:
    os.makedirs(f'./data/savemesh/')
except FileExistsError:
    pass

In [ ]:
my_hucs.head(1)

In [ ]:
import os

os.makedirs("./shapes", exist_ok=True)

for _, row in my_hucs.iterrows():
  huc_id = row.get("huc12", row.get("huc", row.get("ID")))
  if pd.isna(huc_id):
      continue
  one = my_hucs.loc[[row.name]].copy()
  one.to_file(f"./shapes/{huc_id}.shp", engine="pyogrio")


In [ ]:
sum_watershed_areaSqkm = 0.0

for _, row in my_hucs.iterrows():
  area_sqkm = row["areasqkm"]
  print(f"Area of huc {row['huc12']}: {area_sqkm}")
  sum_watershed_areaSqkm += area_sqkm

sum_watershed_areaSqkm = round(sum_watershed_areaSqkm, 1)
print(f"Watershed Area in Sq Km is {sum_watershed_areaSqkm}")

watershedArea_m2 = sum_watershed_areaSqkm * 1_000_000
print(f"Watershed Area in Sq m is {watershedArea_m2}")


In [ ]:
area_sq = sum_watershed_areaSqkm
area_sq

In [ ]:
outputs

In [ ]:
import os
import geopandas as gpd

os.makedirs("./shapes", exist_ok=True)

outputs['watershed_shapefile_filename'] = f'./shapes/{name}_bounds.shp'

watershed_boundary = gpd.GeoDataFrame(
  {"id": [123]},
  geometry=[watershed.exterior],
  crs=crs,
)

watershed_boundary.to_file(
  outputs['watershed_shapefile_filename'],
  engine="pyogrio",
)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Load the shapefile
shapefile_path = outputs['watershed_shapefile_filename']
gdf = gpd.read_file(shapefile_path)

# Plot only the boundaries
fig, ax = plt.subplots(figsize=(10, 10))
gdf.boundary.plot(ax=ax, color='black', linewidth=1) 

# Customize the plot
ax.set_title("Shapefile Boundaries")
plt.show()


In [ ]:
from ipywidgets import interact, widgets
%matplotlib widget

# activate this piece to remove the zoom-in
%matplotlib inline


## Generate the surface mesh

First we'll generate the flattened, 2D triangulation, which builds on hydrography data.  Then we download a digital elevation map from the National Elevation Dataset, and extrude that 2D triangulation to a 3D surface mesh based on interpolation between pixels of the DEM.

### Get river network

This will download the river network from the NHD Plus database, and simplify the network, constructing a tree-like data structure.

In [ ]:
if include_rivers:
  reaches = sources['hydrography'].getShapesByGeometry(
      watershed.exterior, crs, out_crs=crs)

  rivers = watershed_workflow.river_tree.createRivers(reaches, method='hydroseq')

  rivers = watershed_workflow.reduceRivers(
      rivers,
      ignore_small_rivers=ignore_small_rivers,
      prune_by_area=prune_by_area_fraction * watershed.exterior.area * 1.0e-6,
      remove_diversions=True,
      remove_braided_divergences=True,
  )
else:
  reaches = []
  rivers = []

In [ ]:
rivers

In [ ]:
import copy

rivers_orig = [river.deepcopy() for river in rivers]
watershed_orig = copy.deepcopy(watershed)

if generate_plots:
  fig, ax = plt.subplots(figsize=(30, 20))

  watershed_orig.plot(ax=ax, color="k")

  x_coords = []
  y_coords = []
  for river in rivers_orig:
      for node in river.preOrder():
          x, y = node.linestring.xy
          x_coords.extend(x)
          y_coords.extend(y)

  ax.scatter(x_coords, y_coords, color='blue', s=10)

  ax.set_title('Right after getting from the NHD Plus data')
  ax.set_aspect('equal')
  plt.show()


In [ ]:
# plot the rivers and watershed
def plot(ws, rivs, ax=None):
    if ax is None:
        fig, ax = plt.subplots(1, 1)
    ws.plot(color='k', marker='+', markersize=10, ax=ax)
    for river in rivs:
        river.plot(marker='x', markersize=10, ax=ax)

plot(watershed, rivers)

In [ ]:
from ipywidgets import interact, widgets
%matplotlib widget

# activate this piece to remove the zoom-in
%matplotlib inline


In [ ]:
watershed_workflow.simplify(
  watershed,
  rivers,
  reach_segment_target_length=refine_L0,
  huc_segment_target_length=refine_L1,
  river_close_distance=refine_d0,
  river_far_distance=refine_d1,
  snap_triple_junctions_tol=150,
  #min_angle=min_angle,
)

In [ ]:
rivers_simplified=[river.deepcopy() for river in rivers] 
watershed_simplified=copy.deepcopy(watershed) 
print('number of reaches in original', len(rivers_orig[0]), 'number of reaches in simplified', len(rivers[0]))

In [ ]:
# ETC: NOTE -- can this be moved into the simplify call?
for river in rivers:
    river.resetDataFrame()

# Now that the river network is set, find the watershed boundary outlets
for river in rivers:
    watershed_workflow.hydrography.findOutletsByCrossings(watershed, river)

In [ ]:
rivers_simplified_v2=[river.deepcopy() for river in rivers] 
watershed_simplified_v2=copy.deepcopy(watershed) 
print('number of reaches in original', len(rivers_orig[0]), 'number of reaches in simplified', len(rivers[0]))

In [ ]:
shapefile_path = outputs['watershed_shapefile_filename']
gdf_bounds = gpd.read_file(shapefile_path)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(15, 10))

# Plot river nodes/vertices
for river in rivers:
  for node in river.preOrder():
      x, y = node.linestring.xy
      ax.plot(x, y, 'o', markersize=1, markerfacecolor='b',
markeredgecolor='green')

# Plot original watershed geometry
watershed_orig.plot(ax=ax, color='k')

# Plot original flow network if it exists
# Plot watershed boundary shapefile
gdf_bounds.boundary.plot(ax=ax, color='black', zorder=1, lw=3)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.axis('off')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
meshsize

In [ ]:
factor

In [ ]:
if generate_plots:
  fig, axs = plt.subplots(2, 1, figsize=figsize)

  watershed_orig.plot(ax=axs[0], color='k')
  axs[0].set_title('original river network and hucs', fontsize=10)

  watershed_simplified.plot(ax=axs[1], color='k')
  axs[1].set_title('after simplify and prune', fontsize=10)

  for river in rivers_orig:
      for node in river.preOrder():
          x, y = node.linestring.xy
          axs[0].plot(x, y, '-o', markersize=1)

  for river in rivers_simplified:
      for node in river.preOrder():
          x, y = node.linestring.xy
          axs[1].plot(x, y, '-o', markersize=1)

  for ax in axs:
      ax.set_aspect('equal')

  plt.show()

In [ ]:
plot(watershed, rivers)

In [ ]:
watershed.df.columns

In [ ]:
rivers=[river.deepcopy() for river in rivers_simplified] 
watershed=copy.deepcopy(watershed_simplified)

In [ ]:
if generate_plots:
  fig, axs = plt.subplots(3, 1, figsize=(figsize[0], figsize[0] * 1.5))

  for huc in watershed_orig.polygons():
      axs[0].plot(huc.exterior.xy[0], huc.exterior.xy[1], 'k-x', markersize=1)
  axs[0].set_title('original river network and hucs', fontsize=10)

  for huc in watershed_simplified.polygons():
      axs[1].plot(huc.exterior.xy[0], huc.exterior.xy[1], 'k-x',markersize=1)
  axs[1].set_title('after simplify and prune', fontsize=10)

  for river in rivers_orig:
      for node in river.preOrder():
          x, y = node.linestring.xy
          axs[0].plot(x, y, '-o', markersize=1)

  for river in rivers_simplified:
      for node in river.preOrder():
          x, y = node.linestring.xy
          axs[1].plot(x, y, '-o', markersize=1)

  for ax in axs:
      ax.set_aspect('equal')
      ax.axis('off')

  plt.tight_layout()
  plt.show()

### Generate meshes using river network and watershed shape

Triangulation refinement: refine triangles if their area (in m^2) is greater than A(d), where d is the 
distance from the triangle centroid to the nearest stream.  A(d) is a piecewise linear function -- A = A0 if d <= d0, A = A1 if d >= d1, and linearly interpolates between the two endpoints.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Extract stream orders from river tree
stream_ordersX = []
for tree in rivers:
    stream_ordersX.extend([r.properties["stream_order"] for r in tree.preOrder()])

stream_order_counts = Counter(stream_ordersX)

# Print the counts
for order, count in sorted(stream_order_counts.items()):
    print(f"Stream Order {order}: {count} segments")

# Plot the distribution
orders = list(stream_order_counts.keys())
counts = list(stream_order_counts.values())

plt.figure(figsize=(8, 5))
plt.bar(orders, counts, width=0.6)
plt.xlabel("Stream Order")
plt.ylabel("Number of River Segments")
plt.title("Distribution of Stream Orders")
plt.grid(axis='y')
plt.show()


In [ ]:
stream_orders = set()
for tree in rivers:
    stream_orders.update(int(r.properties["stream_order"]) for r in tree.preOrder())
stream_orders 

In [ ]:
widths = [2,4,8,12]

In [ ]:
stream_orders

In [ ]:
riverwidths = dict(zip(stream_orders, widths))
riverwidths

In [ ]:
def river_width(reach):
  order = int(reach.properties["stream_order"])
  return riverwidths.get(order, widths[-1])

In [ ]:
k_factor

In [ ]:
meshsize

In [ ]:
factor

In [ ]:

if generate_plots:
  fig, ax = plt.subplots(figsize=(30, 20))

  # Plot watershed polygons/boundaries
  watershed.plot(ax=ax, color='k')

  x_coords = []
  y_coords = []

  for river in rivers:
      for node in river.preOrder():
          x, y = node.linestring.xy
          x_coords.extend(x)
          y_coords.extend(y)

  ax.scatter(x_coords, y_coords, color='blue', s=10)
  ax.set_title('Right after getting from the NHD Plus data')

  ax.set_aspect('equal')
  plt.show()


In [ ]:
factor

In [ ]:
meshsize

In [ ]:
refine_d0 = meshsize * (factor - 1) if factor >= 2 else meshsize
refine_d1 = meshsize * (factor - 1) * 3 if factor >= 2 else meshsize * 3
refine_A0 = (meshsize**2) / 2
refine_A1 = (np.round(float(meshsize * factor * k_factor)))**2 / 2

d0 = refine_d0
d1 = refine_d1
A0 = refine_A0
A1 = refine_A1

min_angle = 32

m2, areas, dists = watershed_workflow.tessalateRiverAligned(
  watershed,
  rivers,
  river_width=river_width,
  refine_min_angle=min_angle,
  refine_distance=[d0, A0, d1, A1],
  diagnostics=True)

print('total area =', np.sum(areas), 'm^2')
print(my_hucs.geometry.area.sum())
print(watershed.exterior.area)

In [ ]:
mesh_points2 = m2.coords
conn_list = m2.conn

In [ ]:
len(conn_list)

In [ ]:
m2.conn[0]

### Map mesh to DEM

Download a DEM from USGS NED and elevate the triangle nodes to the DEM.

In [ ]:
# get a raster for the elevation map, based on 3DEP
dem = sources['DEM'].getDataset(watershed.exterior, watershed.crs)['dem']

In [ ]:
# Plot the DEM raster
fig, ax = plt.subplots(1,1)

# Plot the DEM data
im = dem.plot(ax=ax, cmap='terrain', add_colorbar=False)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Elevation (m)', rotation=270, labelpad=15)

gdf_bounds_proj = copy.deepcopy(gdf_bounds).to_crs(dem.rio.crs)
gdf_bounds_proj.boundary.plot(ax=ax, color='k', zorder=1,
lw=0.25)

# Add title and labels
ax.set_title('Digital Elevation Model (DEM)', fontsize=14, fontweight='bold')
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')

# Set equal aspect ratio
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

#### Save the DEM as tif

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(30, 20))
dem.plot(ax=ax, cmap='terrain', cbar_kwargs={'label': 'Elevation (m)',
'orientation': 'horizontal'})
ax.set_title('Digital Elevation Model (DEM)')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.show()

"""
import os
output_path = f"./data/dem/{name}_source_DEM_3D_DEP_13arcsec_March30_2026.tif"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
dem.rio.to_raster(output_path)
print(f"DEM saved to {output_path}")
"""

#### DEM Smoothening:

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import scipy.ndimage

dem_orig_copy = dem.copy()

dem_arr = dem.values

if num_smoothing_sweeps > 0:
  dem_sm_arr = scipy.ndimage.gaussian_filter(dem_arr,
sigma=num_smoothing_sweeps, mode='nearest')
else:
  dem_sm_arr = dem_arr.copy()

dem_sm = xr.DataArray(
  dem_sm_arr,
  coords=dem.coords,
  dims=dem.dims,
  attrs=dem.attrs,
  name=dem.name,
)

gdf_bounds_proj = copy.deepcopy(gdf_bounds).to_crs(dem.rio.crs)

xmin, ymin, xmax, ymax = dem.rio.bounds()
extent = [xmin, xmax, ymin, ymax]

fraction, pad = 0.046, 0.12
fig, axs = plt.subplots(1, 3, figsize=(14, 3))

im0 = axs[0].imshow(dem_arr, extent=extent, origin='upper',
aspect='auto')
axs[0].set_title('original DEM')
fig.colorbar(im0, ax=axs[0], orientation='horizontal', fraction=fraction,
pad=pad, label='[m]')
gdf_bounds_proj.boundary.plot(ax=axs[0], color='purple', zorder=1,
lw=0.25)

im1 = axs[1].imshow(dem_sm_arr, extent=extent, origin='upper',
aspect='auto')
axs[1].set_title('smoothed DEM (gaussian filter)')
fig.colorbar(im1, ax=axs[1], orientation='horizontal', fraction=fraction,
pad=pad, label='[m]')

im2 = axs[2].imshow(dem_sm_arr - dem_arr, extent=extent, origin='upper',
aspect='auto')
axs[2].set_title('smoothed DEM - original DEM')
fig.colorbar(im2, ax=axs[2], orientation='horizontal', fraction=fraction,
pad=pad, label='[m]')

plt.tight_layout()
plt.show()

### Elevate and Condition the Mesh: Assign DEM to the mesh

In [ ]:
dem_smooth = False
if dem_smooth: 
    dem = dem_sm

# provide surface mesh elevations
watershed_workflow.elevate(m2, dem, method='linear')

# also elevate the river network linestrings
watershed_workflow.condition.setProfileByDEM(rivers, dem)

### Hydrologic Conditioning of the Mesh

There are a range of options to condition river corridor mesh. We hydrologically condition the river mesh, ensuring unimpeded water flow in river corridors by globally adjusting flowlines to rectify artificial obstructions from inconsistent DEM elevations or misalignments. Please read the documentation for more information

In the pit-filling algorithm, we want to make sure that river corridor is not filled up. Hence we exclude river corridor cells from the pit-filling algorithm.

In [ ]:
# now condition the river to fix mis-hits, where the corridor centroids do not fall in the DEM's idea of the river, enforcing monotonicity of the river network
def computeBurnInDepth(da_sq_miles):
    """burn-in depth as a function of drainage area"""
    depth_in_feet = 1.22 * da_sq_miles**0.317
    return 0.3048 * depth_in_feet # ft --> meters


def computeBurnInDepthFromReach(reach):
    depth = computeBurnInDepth(reach['drainage_area_sqkm'] * 0.386102)
    logging.debug(f"reach of DA {reach['drainage_area_sqkm']} has depth {depth}")
    return depth
    

watershed_workflow.condition.conditionRiverMeshes(m2,
                       rivers,
                       network_burn_in_depth=computeBurnInDepthFromReach)

In [ ]:
# hydrologically condition the non-corridor portion of the mesh, removing pits
outlet_edge = watershed_workflow.mesh.Edge(rivers[0]['elems'][-1][0], rivers[0]['elems'][-1][-1])
preserved_pits = [c for (c,conn) in enumerate(m2.conn) if len(conn) > 3]

m2r, res = watershed_workflow.condition.conditionMesh(m2,
                                                      preserved_pits=preserved_pits,
                                                      forced_outlet_edges=[outlet_edge,],
                                                      epsilon = 0.01
                                                     )

In [ ]:
# plotting surface mesh with elevations
fig, ax = plt.subplots(1,1)
ax2 = ax.inset_axes([0.53,0.04,0.2,0.25])

mp = m2.plot(facecolors='elevation', edgecolors=None, ax=ax, linewidth=0.5, colorbar=False)
cbar = fig.colorbar(mp, orientation="horizontal")
ax.set_title('surface mesh with elevations')
ax.set_aspect('equal', 'datalim')

mp2 = m2.plot(facecolors='elevation', edgecolors='white', ax=ax2, colorbar=False)
ax2.set_aspect('equal', 'datalim')

xlim = (-1.52e6, -1.51975e6)
ylim = (638450, 638460)

ax2.set_xlim(xlim)
ax2.set_ylim(ylim)
ax2.set_xticks([])
ax2.set_yticks([])

ax.indicate_inset_zoom(ax2, edgecolor='k')

cbar.ax.set_title('elevation [m]')

plt.show()
# [1445055.35066667 -646519.51766667]

### Plot the mesh and save the mesh as an ESRI Shapefile

In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon

verts = m2.coords[:, :2]        
conn = m2.conn                   
mesh_polygons = []
for cell_nodes in conn:
    valid_nodes = [n for n in cell_nodes if n != -1]
    polygon = Polygon(verts[valid_nodes])
    mesh_polygons.append(polygon)
# Extract centroid coordinates
centroids = m2.centroids
centroid_x = centroids[:, 0]
centroid_y = centroids[:, 1]
centroid_z = centroids[:, 2]
# Create GeoDataFrame
gdf_mesh = gpd.GeoDataFrame({
    'MeshID': range(len(mesh_polygons)),
    'centroid_x [m]': centroid_x,
    'centroid_y [m]': centroid_y,
    'centroid_z [m]': centroid_z,
}, geometry=mesh_polygons, crs=m2.crs)
gdf_mesh['area [m2]'] = gdf_mesh.geometry.area

meshfilename = f'./shapes/early_mesh{choose_mesh_tag_name}_m{meshsize}_f{factor}_k{k_factor_text}.shp'
gdf_mesh.to_file(meshfilename)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
gdf_mesh.plot(ax=ax, facecolor = 'none', edgecolor = 'gray', lw  = 0.5)
ax.set_title(f'm{meshsize}_f{factor}_k{k_factor_text}')

In [ ]:
sum(gdf_mesh['area [m2]'])/(1e6)

### Waterbodies: Obtain Waterbodies directly from NHD (addition to current WW)

In [ ]:

  import geopandas as gpd
  import pandas as pd
  import shapely
  from pynhd import NHDPlusHR

  prune_by_area_fraction_waterbodies = 0

  # 1. Build query geometry in lat/lon, because HyRiver expects that cleanly
  query_gdf = gpd.GeoDataFrame(geometry=[watershed.exterior],
  crs=crs).to_crs("EPSG:4269")

  # 2. Query NHDPlus HR waterbodies by geometry
  waterbodies = NHDPlusHR("waterbody").bygeom(query_gdf.geometry.iloc[0],
  "EPSG:4269")

  # 3. Add a usable ID column
  id_candidates = [
      "nhdplusid",
      "permanent_identifier",
      "comid",
      "featureid",
      "gnis_id",
  ]
  for col in id_candidates:
      if col in waterbodies.columns:
          waterbodies["ID"] = waterbodies[col].astype("string")
          break

  if "ID" not in waterbodies.columns:
      waterbodies["ID"] = waterbodies.index.astype("string")

  # 4. Reproject back to your working CRS
  waterbodies = waterbodies.to_crs(crs)

  # 5. Keep only features intersecting the watershed and clip them
  waterbodies =waterbodies[waterbodies.intersects(watershed.exterior)].copy()
  waterbodies["geometry"] =waterbodies.geometry.intersection(watershed.exterior)
  waterbodies = waterbodies[~waterbodies.geometry.is_empty].copy()

  # 6. Optional area pruning
  if prune_by_area_fraction_waterbodies is not None:
      min_area = prune_by_area_fraction_waterbodies * watershed.exterior.area
      waterbodies = waterbodies[waterbodies.geometry.area >=min_area].copy()

  print("count:", len(waterbodies))
  print(waterbodies.columns)
  display(waterbodies.head())

  #Quick sanity plot:
  waterbodies["watershed_area_fraction"] = waterbodies["areasqkm"] / (sum(gdf_mesh['area [m2]'])/(1e6))

  fig, ax = plt.subplots(figsize=(12, 10))
  gpd.GeoDataFrame(geometry=[watershed.exterior],
  crs=crs).boundary.plot(ax=ax, color="black", lw=1)

  if len(waterbodies) > 0:
      waterbodies.plot(ax=ax, color="cornflowerblue", edgecolor="navy", alpha=0.7)
      ax.set_title(f'All Waterbodies in the {name} Watershed', fontsize =15)
      
      print(waterbodies[["gnis_name", "areasqkm", "fcode", "watershed_area_fraction"]])
  ax.set_aspect("equal")
  plt.show()


In [ ]:
waterbodies

#### We have waterbodies but current WW v2.0 has no way to handle it apparently; so far we can only create and mask it but dont know what to do with it; 

As this #watershed_workflow.condition.fill_pits_dual(m2, is_waterbody=waterbody_mask) does not exist. condition.fill_pits_dual does not work; 

In [ ]:
# hydrologically condition the mesh, removing pits.
# replace conditioned mesh where there are water bodies!

if len(waterbodies) > 0:
  all_bodies = shapely.union_all(waterbodies.geometry)
  centroids = np.asarray(m2.centroids)
  centroid_points = shapely.points(centroids[:, 0], centroids[:, 1])
  waterbody_mask = shapely.contains(all_bodies, centroid_points).astype(int)
else:
  waterbody_mask = np.zeros(len(m2.conn), dtype=int)

print(waterbody_mask.shape)
print(waterbody_mask.sum())

### THE following has no use in ww v2.0 (March 30, 2026)

# NOTE: this fills reservoirs as well!  Might have to think about how to allow some pits!
#watershed_workflow.condition.fill_pits_dual(m2, is_waterbody=waterbody_mask)

##### Add the watershed boundary, HUCs and outlet

In [ ]:
stream_outlet_width

In [ ]:
import watershed_workflow.hydrography as ww_hydro
import watershed_workflow.sources.standard_names as names
import shapely

watershed.df["name_orig"] = watershed.df[names.NAME]
watershed.df[names.NAME] = watershed.df["huc12"].astype(str)

ww_hydro.findOutletsByHydroseq(watershed, rivers[0], tol=0.0)

watershed.exterior_outlet = shapely.Point(watershed.exterior_outlet.coords[0][0:2])
watershed.df[names.OUTLET] = [
    None if pt is None else shapely.Point(pt.coords[0][0:2])
    for pt in watershed.df[names.OUTLET]
]

In [ ]:
# add labeled sets for subcatchments and outlets
watershed_workflow.regions.addWatershedAndOutletRegions(
    m2,
    watershed,
    outlet_width=stream_outlet_width,
    exterior_outlet=True,
)

# add labeled sets for river corridor cells
watershed_workflow.regions.addRiverCorridorRegions(m2, rivers)

# add labeled sets for river corridor cells by order
watershed_workflow.regions.addStreamOrderRegions(m2, rivers)

In [ ]:
for ls in m2.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import shapely
import watershed_workflow.sources.standard_names as names

watershed_boundary_gdf = gpd.GeoDataFrame(
    geometry=[watershed.exterior.boundary],
    crs=crs,
)

if gdf_mesh.crs != crs:
    gdf_mesh_plot = gdf_mesh.to_crs(crs)
else:
    gdf_mesh_plot = gdf_mesh

outlet_face_lines = []
outlet_face_labels = []

for ls in m2.labeled_sets:
    if ls.entity == "FACE" and "outlet" in ls.name.lower():
        for e in ls.ent_ids:
            p0 = m2.coords[e[0]][:2]
            p1 = m2.coords[e[1]][:2]
            outlet_face_lines.append(shapely.LineString([p0, p1]))
            outlet_face_labels.append(ls.name)

outlet_faces_gdf = gpd.GeoDataFrame(
    {"label": outlet_face_labels},
    geometry=outlet_face_lines,
    crs=crs,
)

domain_outlet_faces = outlet_faces_gdf[outlet_faces_gdf["label"] == "surface domain outlet"].copy()
polygon_outlet_faces = outlet_faces_gdf[outlet_faces_gdf["label"] != "surface domain outlet"].copy()

exterior_outlet_gdf = gpd.GeoDataFrame(
    {"label": ["exterior_outlet"]},
    geometry=[shapely.Point(watershed.exterior_outlet.coords[0][:2])],
    crs=crs,
)

fig, ax = plt.subplots(figsize=(12, 10))

gdf_mesh_plot.boundary.plot(ax=ax, color="lightgray", lw=0.3)
watershed_boundary_gdf.plot(ax=ax, color="black", linewidth=1.5)

if len(domain_outlet_faces) > 0:
    domain_outlet_faces.plot(ax=ax, color="red", linewidth=5)

if len(polygon_outlet_faces) > 0:
    polygon_outlet_faces.plot(ax=ax, color="cyan", linewidth=2.5)

exterior_outlet_gdf.plot(ax=ax, color="red", markersize=70)

x, y = exterior_outlet_gdf.geometry.iloc[0].x, exterior_outlet_gdf.geometry.iloc[0].y
pad = 160
ax.set_xlim(x - pad, x + pad)
ax.set_ylim(y - pad, y + pad)


from matplotlib.lines import Line2D

legend_handles = [
    Line2D([0], [0], color="red", lw=4, label="surface domain outlet"),
    Line2D([0], [0], color="cyan", lw=2.5, label="polygon outlet edges"),
    Line2D([0], [0], marker="o", color="red", linestyle="None", markersize=8, label="exterior outlet point"),
]
ax.legend(handles=legend_handles, loc="best")

ax.set_title("Exterior outlet and polygon outlet faces")
ax.set_aspect("equal")
plt.show()

### dsiplay the polygons
display_cols = [names.NAME]
if "huc12" in watershed.df.columns and "huc12" not in display_cols:
    display_cols.append("huc12")
if names.OUTLET in watershed.df.columns:
    display_cols.append(names.OUTLET)

print("Polygons shown in this watershed/outlet map:")
display(watershed.df[display_cols].copy())

In [ ]:
data = [[meshsize, factor, k_factor_text, refine_A0, refine_d0, refine_A1, refine_d1,  np.round(np.min(areas),2), np.round(np.max(areas),2), len(m2.conn)]]
columns = ['Meshsize', 'Factor', 'k_factor', 'Refine_A0', 'Refine_d0', 'Refine_A1', 'Refine_d1', 'Min_Area', 'Max_Area', 'TotalSuf_mesh']
df = pd.DataFrame(data, columns=columns)

df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(15, 10))

gdf_mesh.plot(
    ax=ax,
    column="centroid_z [m]",
    cmap="pink",
    linewidth=0.25,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "elevation [m]", "orientation": "horizontal"},
)

for river in rivers:
    for node in river.preOrder():
        x, y = node.linestring.xy
        ax.plot(x, y, color="b", linewidth=0.05, ls="--")

gdf_bounds.boundary.plot(ax=ax, color="purple", lw=0.25, zorder=1)

ax.set_aspect("equal")
ax.set_title("Surface mesh with elevations")
plt.show()

#fig.savefig(f'./images/{name}/{name}_mesh_m{meshsize}_f{factor}_k{k_factor_text}', dpi = 1200)

### Finally Save the River shapefile

In [ ]:
rivers_sim_den_pru = [river.deepcopy() for river in rivers]

from shapely.geometry import LineString
import geopandas as gpd
import watershed_workflow.sources.standard_names as names

features = []
for river in rivers_sim_den_pru:
    for node in river.preOrder():
        x, y = node.linestring.xy
        line = LineString(zip(x, y))

        stream_order = node.properties[names.ORDER] if names.ORDER in node.properties else -1
        river_id = node.properties[names.ID] if names.ID in node.properties else None

        features.append({
            "geometry": line,
            "stream_order": stream_order,
            "length": line.length,
            "id": river_id,
        })

gdf_sim_den_rivers = gpd.GeoDataFrame(features, crs=crs)
gdf_sim_den_rivers.to_file(
    f"./shapes/{name}_rivers_sim_den_prunned_m{meshsize}_f{factor}_k{k_factor_text}.shp",
    engine="pyogrio",
)


## Surface properties

Meshes interact with data to provide forcing, parameters, and more in the actual simulation.  Specifically, we need vegetation type on the surface to provide information about transpiration and subsurface structure to provide information about water retention curves, etc.

We'll start by downloading and collecting land cover from the NLCD dataset, and generate sets for each land cover type that cover the surface.  Likely these will be some combination of grass, deciduous forest, coniferous forest, and mixed forest.

### NLCD LandCover From the Tempalte

#### Lets see the percentage distribution of the NLCD land cover types inside the watershed mesh

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from watershed_workflow.sources.manager_nlcd import colors as nlcd_color_info

# get NLCD cover raster in working CRS
lc_ds = sources["land cover"].getDataset(
    watershed.exterior.buffer(100),
    crs,
    variables=["cover"],
    out_crs=crs,
)

nlcd = lc_ds["cover"]

# sample land-cover values at mesh centroids
centroids = m2.centroids[:, :2]
lc = nlcd.sel(
    x=xr.DataArray(centroids[:, 0], dims="points"),
    y=xr.DataArray(centroids[:, 1], dims="points"),
    method="nearest",
).values.squeeze()

# standard WW colormap
nlcd_indices, nlcd_cmap, nlcd_norm, nlcd_ticks, nlcd_labels = \
    watershed_workflow.colors.createNLCDColormap(np.unique(lc))

# histogram / counts
u, count = np.unique(lc, return_counts=True)
count_sort_ind = np.argsort(-count)
u_sorted = u[count_sort_ind]
count_sorted = count[count_sort_ind]

lc_dict = dict(zip(u_sorted, count_sorted))
print(lc_dict)

plt.hist(lc, bins="auto")
plt.ylabel("counts")
plt.xlabel("index")
plt.show()

# summary table
df_lc = pd.DataFrame({
    "NLCD_code": u_sorted,
    "NLCD_label": [nlcd_color_info[int(code)][0] if int(code) in nlcd_color_info else "Unknown" for code in u_sorted],
    "n_mesh_cells": count_sorted,
    "percent_mesh_cells": 100 * count_sorted / np.sum(count_sorted),
})

display(df_lc)


#### Following the template

In [ ]:
# download the NLCD raster
nlcd = sources['land cover'].getDataset(watershed.exterior.buffer(100), watershed.crs)['cover']
# what land cover types did we get?
logging.info('Found land cover dtypes: {}'.format(nlcd.dtype))
logging.info('Found land cover types: {}'.format(set(list(nlcd.values.ravel()))))

In [ ]:
# create a colormap for the data
nlcd_indices, nlcd_cmap, nlcd_norm, nlcd_ticks, nlcd_labels = \
      watershed_workflow.colors.createNLCDColormap(np.unique(nlcd))
nlcd_cmap

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(18, 8))
nlcd.plot.imshow(ax=ax, cmap=nlcd_cmap, norm=nlcd_norm, add_colorbar=False)
watershed_workflow.colors.createIndexedColorbar(
    ncolors=len(nlcd_indices),
    cmap=nlcd_cmap,
    labels=nlcd_labels,
    ax=ax,
)

ax.set_title("NLCD land cover index", fontsize=22)
ax.axis("off")

cbar_ax = fig.axes[-1]
cbar_ax.tick_params(labelsize=18)

plt.tight_layout()

fig.savefig(
    f'./images/{name}/{name}_land_cover_raw_m{meshsize}_f{factor}_k{k_factor_text}',
    dpi=1200,
    bbox_inches="tight",
    pad_inches=0.1,
)

plt.show()


In [ ]:
## we dont need all the nlcd classes; lets group some of them to preserve the dominant signals

# keep selected classes, merge everything else into "Other" = 99
nlcd_color_new = 99 * np.ones_like(lc)

groupings = {
    #41 : ["Deciduous Forest", 
    #        "Mixed Forest", 
    #        "Woody Wetlands"
    #     ],
    42: ["Evergreen Forest"],
    52: [
        "Dwarf Scrub",
        "Shrub/Scrub",
        "Sedge/Herbaceous",
        "Pasture/Hay",
        "Cultivated Crops",
    ],
    71: ["Grassland/Herbaceous"],
}

for k, v in groupings.items():
    for label in v:
        index = sources["land cover"].indices[label]
        nlcd_color_new[np.where(lc == index)] = k

# histogram / counts
u, count = np.unique(nlcd_color_new, return_counts=True)
count_sort_ind = np.argsort(-count)
u_sorted = u[count_sort_ind]
count_sorted = count[count_sort_ind]

print(u_sorted)
print(count_sorted)

aa = count_sorted
print(aa[0] / np.sum(aa) * 100, aa[1] / np.sum(aa) * 100, aa[2] / np.sum(aa) * 100)

plt.hist(nlcd_color_new, bins="auto")
plt.ylabel("counts")
plt.xlabel("index")
plt.show()

# derive labels from existing NLCD labels where possible
df_lc_simple = pd.DataFrame({
    "NLCD_code_grouped": u_sorted,
    "NLCD_label_grouped": [
        sources["land cover"].colors[int(code)][0] if int(code) in sources["land cover"].colors else "Other"
        for code in u_sorted
    ],
    "n_mesh_cells": count_sorted,
    "percent_mesh_cells": 100 * count_sorted / np.sum(count_sorted),
})

display(df_lc_simple)


In [ ]:
# simplified NLCD already aligned to mesh cells
m2_nlcd = nlcd_color_new.copy()
m2.cell_data["land_cover"] = m2_nlcd

assert 127 not in m2_nlcd

# derive labels automatically from the representative NLCD codes
nlcd_labels_dict = {
    code: sources["land cover"].colors[int(code)][0]
    for code in groupings.keys()
}
nlcd_labels_dict[99] = "Other"

# keep only labels that actually appear on the mesh
nlcd_labels_dict = {
    k: v for k, v in nlcd_labels_dict.items()
    if k in np.unique(m2_nlcd)
}

watershed_workflow.regions.addSurfaceRegions(
    m2,
    column="land_cover",
    names=nlcd_labels_dict,
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

nlcd_indices = list(np.unique(m2_nlcd))
nlcd_labels = [nlcd_labels_dict[int(code)] for code in nlcd_indices]

nlcd_colors = []
for code in nlcd_indices:
    if int(code) in sources["land cover"].colors:
        nlcd_colors.append(sources["land cover"].colors[int(code)][1])
    else:
        nlcd_colors.append((0.6, 0.6, 0.6))  # Other

nlcd_cmap = mcolors.ListedColormap(nlcd_colors)

code_to_idx = {code: i for i, code in enumerate(nlcd_indices)}
m2_nlcd_plot = np.array([code_to_idx[v] for v in m2_nlcd])

nlcd_norm = mcolors.BoundaryNorm(
    np.arange(len(nlcd_indices) + 1) - 0.5,
    nlcd_cmap.N,
)

m2.plot(
    ax=ax,
    facecolors=m2_nlcd_plot,
    cmap=nlcd_cmap,
    norm=nlcd_norm,
    edgecolors=None,
    colorbar=False,
)

watershed_workflow.colors.createIndexedColorbar(
    ncolors=len(nlcd_indices),
    cmap=nlcd_cmap,
    labels=nlcd_labels,
    ax=ax,
)

ax.set_title("NLCD Land Cover Index - Post Grouping", fontsize=22)
ax.set_axis_off()
cbar_ax = fig.axes[-1]
cbar_ax.tick_params(labelsize=18)

fig.savefig(
    f'./images/{name}/{name}_land_cover_post_grouping_m{meshsize}_f{factor}_k{k_factor_text}',
    dpi=1200,
    bbox_inches="tight",
    pad_inches=0.1,
)

plt.show()


In [ ]:
nlcd_labels_dict

In [ ]:
for ls in m2.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

## Subsurface properties

### NRCS Soils

In [ ]:
# get NRCS shapes, on a reasonable crs

import watershed_workflow.sources.manager_nrcs as manager_nrcs
sources['soil structure'] = manager_nrcs.ManagerNRCS(force_download=True)
nrcs = sources['soil structure'].getShapesByGeometry(watershed.exterior, watershed.crs, out_crs=crs)


In [ ]:
nrcs

In [ ]:
#saving the nrcs soil prop right after getShapesbyGeometry
nrcs.to_csv(f"./storecsv/{name}/{name}_nrcs_right_after_props_getShapesbyGeometry_m{meshsize}_f{factor}_k{k_factor_text}.csv", index=False)

In [ ]:
nrcs.columns

In [ ]:
nrcs[[
    "Rosetta permeability [m^2]",
    "permeability [m^2]",
    "bulk density [g/cm^3]",
    "total sand pct [%]",
    "total silt pct [%]",
    "total clay pct [%]",
]].describe()


In [ ]:
nrcs.sort_values("Rosetta permeability [m^2]", ascending=False)[[
    "mukey",
    "Rosetta permeability [m^2]",
    "permeability [m^2]",
    "bulk density [g/cm^3]",
    "total sand pct [%]",
    "total silt pct [%]",
    "total clay pct [%]",
]].head(10)

In [ ]:
nrcs_check = nrcs.copy()

poro_fill_mask = nrcs_check["porosity [-]"].isna() & nrcs_check["Rosetta porosity [-]"].notna()
perm_fill_mask = nrcs_check["permeability [m^2]"].isna() & nrcs_check["Rosetta permeability [m^2]"].notna()

print("Rows that would fill porosity from Rosetta:", poro_fill_mask.sum())
print("Rows that would fill permeability from Rosetta:", perm_fill_mask.sum())
print("Rows that would fill either from Rosetta:", (poro_fill_mask | perm_fill_mask).sum())


In [ ]:
"""
print("#### NOTE: WE DROP THIS BECAUSE OF UNUSUAL ROSETTA PERM AND PORO WE ABSOLUTELY CANNOT FALL BACK ON THIS RIGHT NOW ####")
print()
print("Original NRCS rows before requiring native NRCS porosity and permeability:", len(nrcs))

nrcs = nrcs[
    nrcs["porosity [-]"].notna() &
    nrcs["permeability [m^2]"].notna()
].copy()

print("Remaining NRCS rows after requiring native NRCS porosity and permeability:", len(nrcs))
"""

In [ ]:
# create a clean dataframe with just the data we will need for ATS
def replace_column_nans(df, col_nan, col_replacement):
    """In a df, replace col_nan entries by col_replacement if is nan.  In Place!"""
    row_indexer = df[col_nan].isna()
    df.loc[row_indexer, col_nan] = df.loc[row_indexer, col_replacement]
    return

# where native porosity or permeability is missing, fill from Rosetta
replace_column_nans(nrcs, 'porosity [-]', 'Rosetta porosity [-]')
replace_column_nans(nrcs, 'permeability [m^2]', 'Rosetta permeability [m^2]')

# drop unnecessary columns
for col in ['Rosetta porosity [-]', 'Rosetta permeability [m^2]', 'bulk density [g/cm^3]', 'total sand pct [%]',
            'total silt pct [%]', 'total clay pct [%]']:
    nrcs.pop(col)
    
# drop nans
nan_mask = nrcs.isna().any(axis=1)
dropped_mukeys = nrcs.index[nan_mask]

# Drop those rows
nrcs = nrcs[~nan_mask]

assert nrcs['porosity [-]'][:].min() >= min_porosity
assert nrcs['permeability [m^2]'][:].max() <= max_permeability
nrcs

# check for nans
nrcs.isna().any()

In [ ]:
# Compute the soil color of each cell of the mesh
# Note, we use mukey here because it is an int, while ID is a string
soil_color_mukey = watershed_workflow.getShapePropertiesOnMesh(m2, nrcs, 'mukey', 
                                                         resolution=50, nodata=-999)

nrcs.set_index('mukey', drop=False, inplace=True)

unique_soil_colors = list(np.unique(soil_color_mukey))
if -999 in unique_soil_colors:
    unique_soil_colors.remove(-999)

# retain only the unique values of soil_color
nrcs = nrcs.loc[unique_soil_colors]

# renumber the ones we know will appear with an ATS ID using ATS conventions
nrcs['ATS ID'] = range(1000, 1000+len(unique_soil_colors))
nrcs.set_index('ATS ID', drop=True, inplace=True)

# create a new soil color and a soil thickness map using the ATS IDs
soil_color = -np.ones_like(soil_color_mukey)
soil_thickness = np.nan * np.ones(soil_color.shape, 'd')

for ats_ID, ID, thickness in zip(nrcs.index, nrcs.mukey, nrcs['thickness [m]']):
    mask = np.where(soil_color_mukey == ID)
    soil_thickness[mask] = thickness
    soil_color[mask] = ats_ID


m2.cell_data['soil_color'] = soil_color
m2.cell_data['soil thickness'] = soil_thickness


###

## check the missing soil thickness
n_missing_soil_thickness = np.isnan(soil_thickness).sum()
print(f"Number of mesh cells with missing soil thickness: {n_missing_soil_thickness}")
#### check the counts of missing soil color
soil_unassigned_mask = (soil_color == -1)
thickness_unassigned_mask = np.isnan(m2.cell_data["soil thickness"])
print("n soil_color == -1:", soil_unassigned_mask.sum())


In [ ]:
np.mean(m2.cell_data['soil thickness']), np.median(m2.cell_data['soil thickness']),  np.nanmedian(m2.cell_data['soil thickness']), 

In [ ]:
# plot the soil color
# -- get a cmap for soil color
sc_indices, sc_cmap, sc_norm, sc_ticks, sc_labels = \
      watershed_workflow.colors.createIndexedColormap(nrcs.index)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(22, 8))

mp = m2.plot(
    ax=ax,facecolors=m2.cell_data["soil_color"], cmap=sc_cmap,norm=sc_norm,edgecolors=None,colorbar=False,
)

watershed_workflow.colors.createIndexedColorbar(
    ncolors=len(nrcs),cmap=sc_cmap,labels=sc_labels,ax=ax,
)

ax.set_title("NRCS Soil Units on Mesh", fontsize = 22)
ax.set_axis_off()

fig.savefig(
    f'./images/{name}/{name}_NRCS_Soil_Units_on_Mesh_m{meshsize}_f{factor}_k{k_factor_text}',
    dpi=1200,
    bbox_inches="tight",
    pad_inches=0.1,
)

plt.show()


In [ ]:
print("soil_color_mukey:", soil_color_mukey)
print("soil_color_mukey.shape:", soil_color_mukey.shape)
print("len(np.unique(soil_color_mukey)):", len(np.unique(soil_color_mukey)))
print("len(np.unique(soil_color)):", len(np.unique(soil_color)))
print("np.unique(soil_color_mukey):", np.unique(soil_color_mukey))
print("np.unique(soil_color):", np.unique(soil_color))

print()
print()

# inspect soil thickness on the mesh
print("soil_thickness:", soil_thickness)
print("soil_thickness.shape:", soil_thickness.shape)
print("Number of NaNs in soil_thickness:", np.isnan(soil_thickness).sum())
print("np.nanmin(soil_thickness):", np.nanmin(soil_thickness))
print("np.nanmax(soil_thickness):", np.nanmax(soil_thickness))
print("np.nanmedian(soil_thickness) [m] =", np.nanmedian(soil_thickness))
print("len(np.unique(soil_thickness[~np.isnan(soil_thickness)])):",
      len(np.unique(soil_thickness[~np.isnan(soil_thickness)])))
print("unique soil thickness values [m]:",
      np.unique(soil_thickness[~np.isnan(soil_thickness)]))


### Depth to Bedrock from SoilGrids

In [ ]:
dtb = sources['depth to bedrock'].getDataset(watershed.exterior, watershed.crs)['band_1']

# the SoilGrids dataset is in cm --> convert to meters
dtb.values = dtb.values/100.

In [ ]:
# map to the mesh
m2.cell_data['dtb'] = watershed_workflow.getDatasetOnMesh(m2, dtb, method='linear')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(22, 8))

gons = m2.plot(
    ax=ax,
    facecolors=m2.cell_data["dtb"],
    cmap="plasma_r",
    edgecolors=None,
)

cbar = fig.axes[-1]
cbar.tick_params(labelsize=22)
cbar.set_ylabel("[m]", fontsize=22)

ax.set_title("Depth to Bedrock on Mesh", fontsize=22)
ax.set_axis_off()

fig.savefig(
    f'./images/{name}/{name}_DTB_on_Mesh_m{meshsize}_f{factor}_k{k_factor_text}',
    dpi=1200,
    bbox_inches="tight",
    pad_inches=0.1,
)

plt.show()


### GLYHMPS geologic layer

GLYHMPS is complete in that it does not appear to have missing data, but does not have texture properties needed for Water Retention Models.  Instead we rely on scaling laws to fill the data.

In [ ]:
glhymps = sources['geologic structure'].getShapesByGeometry(watershed.exterior.buffer(1000), watershed.crs, out_crs=crs)
glhymps = watershed_workflow.soil_properties.mangleGLHYMPSProperties(glhymps,
                                              min_porosity=min_porosity, 
                                              max_permeability=max_permeability, 
                                              max_vg_alpha=max_vg_alpha)

# intersect with the buffered geometry -- don't keep extras
glhymps = glhymps[glhymps.intersects(watershed.exterior.buffer(10))]
glhymps

In [ ]:
# quality check -- make sure glymps shapes cover the watershed
print(glhymps.union_all().contains(watershed.exterior))
glhymps

In [ ]:
# clean the data
glhymps.pop('logk_stdev [-]')

assert glhymps['porosity [-]'][:].min() >= min_porosity
assert glhymps['permeability [m^2]'][:].max() <= max_permeability
assert glhymps['van Genuchten alpha [Pa^-1]'][:].max() <= max_vg_alpha

glhymps.isna().any()

In [ ]:
glhymps

In [ ]:
# note that for larger areas there are often common regions -- two labels with the same properties -- no need to duplicate those with identical values.
def reindex_remove_duplicates(df, index):
    """Removes duplicates, creating a new index and saving the old index as tuples of duplicate values. In place!"""
    if index is not None:
        if index in df:
            df.set_index(index, drop=True, inplace=True)
    
    index_name = df.index.name

    # identify duplicate rows
    duplicates = list(df.groupby(list(df)).apply(lambda x: tuple(x.index)))

    # order is preserved
    df.drop_duplicates(inplace=True)
    df.reset_index(inplace=True)
    df[index_name] = duplicates
    return

reindex_remove_duplicates(glhymps, 'ID')
glhymps

In [ ]:
# Compute the geo color of each cell of the mesh
geology_color_glhymps = watershed_workflow.getShapePropertiesOnMesh(m2, glhymps, 'index', 
                                                         resolution=50, nodata=-999)

# retain only the unique values of geology that actually appear in our cell mesh
unique_geology_colors = list(np.unique(geology_color_glhymps))
if -999 in unique_geology_colors:
    unique_geology_colors.remove(-999)

# retain only the unique values of geology_color
glhymps = glhymps.loc[unique_geology_colors]

# renumber the ones we know will appear with an ATS ID using ATS conventions
glhymps['ATS ID'] = range(100, 100+len(unique_geology_colors))
glhymps['TMP_ID'] = glhymps.index
glhymps.reset_index(drop=True, inplace=True)
glhymps.set_index('ATS ID', drop=True, inplace=True)

# create a new geology color using the ATS IDs
geology_color = -np.ones_like(geology_color_glhymps)
for ats_ID, tmp_ID in zip(glhymps.index, glhymps.TMP_ID):
    geology_color[np.where(geology_color_glhymps == tmp_ID)] = ats_ID

glhymps.pop('TMP_ID')

m2.cell_data['geology_color'] = geology_color
                            

In [ ]:
geology_color

In [ ]:
m2.cell_data['geology_color']

In [ ]:
geology_color_glhymps.min()

In [ ]:
import numpy as np
import matplotlib.colors as mcolors

# -- get a cmap for geology color
gc_indices, gc_cmap, gc_norm, gc_ticks, gc_labels = \
    watershed_workflow.colors.createIndexedColormap(glhymps.index)

# replace bright green with purple
gc_colors = gc_cmap(np.arange(gc_cmap.N))
gc_colors[1] = mcolors.to_rgba("mediumpurple")   # change index as needed
gc_cmap = mcolors.ListedColormap(gc_colors)

fig, ax = plt.subplots(1, 1, figsize=(22, 8))

gons = m2.plot(
    ax=ax,
    facecolors=m2.cell_data["geology_color"],
    cmap=gc_cmap,
    norm=gc_norm,
    edgecolors=None,
    colorbar=False,
)

watershed_workflow.colors.createIndexedColorbar(
    ncolors=len(glhymps),
    cmap=gc_cmap,
    labels=gc_labels,
    ax=ax,
)

ax.set_title("Geologic Index on Mesh", fontsize=22)
ax.set_axis_off()

cbar_ax = fig.axes[-1]
cbar_ax.tick_params(labelsize=18)


fig.savefig(
    f'./images/{name}/{name}_Geologic_Index_on_Mesh_m{meshsize}_f{factor}_k{k_factor_text}',
    dpi=1200,
    bbox_inches="tight",
    pad_inches=0.1,
)

plt.show()


### Combine to form a complete subsurface dataset

In [ ]:
outputs

In [ ]:
bedrock = watershed_workflow.soil_properties.getDefaultBedrockProperties()

# merge the properties databases
subsurface_props = pd.concat([glhymps, nrcs, bedrock])

# save the properties to disk for use in generating input file

#output_filenames['subsurface_properties'] = toOutput(f'{name}_subsurface_properties.csv')
#subsurface_props.to_csv(output_filenames['subsurface_properties'])

if k_factor:
    outputs['subsurface_properties'] = f'../data-processed/{name}/{name}_sub_prop_m{meshsize}_f{factor}_k{k_factor_text}.csv'
    subsurface_props.to_csv(outputs['subsurface_properties'])
else:
    outputs['subsurface_properties'] = f'../data-processed/{name}/{name}_subsurface_properties.csv'
    subsurface_props.to_csv(outputs['subsurface_properties'])    

subsurface_props



In [ ]:
for ls in m2.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

## A combined, complete product?

As a default, we would like a material-driven (e.g. not fields for porosity, perm, etc, but soil classes, each with a common porosity/permeability/vG curve) default that is valid everywhere.  That makes it clear that we must rely on GLHYMPS as the only material-based product that is valid everywhere.  Other products may be layered on top of this, replacing GLHYMPS values, but the underlying layer should be based on GLHYMPS.  To fill in the van Genuchten properties, we relate alpha to permeability and choose a single common n and s_r.

Where available, we then choose to use SSURGO as a layer on top of GLHYMPS.  So start by using all GLHYMPS values, then override ones where SSURGO is valid with those values.  This will be the second model, and has then three layers -- a bedrock layer, a soil layer from 0 to 2m, and a geologic layer, using GLHYMPS values.  SoilGrids depth-to-bedrock will be used to provide the transition between bedrock and (where > 2m) the GLHYMPS "geologic" layer or (where < 2m) the SSURGO "soil" layer.  Where SSURGO has no values, the underlying GLHYMPS values will be used even in the top 2m.


Note, all integer IDs in mesh files must be unique.  This includes Material IDs, side sets, etc.  We create the Material ID map and data frame.  This is used to standardize IDs from multiple data sources.  Traditionally, ATS numbers Material IDs/Side Sets as:

* 0-9 : reserved for boundaries, surface/bottom, etc
* 10-99 : Land Cover side sets, typically NLCD IDs are used
* 100-999 : geologic layer material IDs. 999 is reserved for bedrock.
* 1000-9999 : soil layer material IDs




## Mesh extrusion

Given the surface mesh and material IDs on both the surface and subsurface, we can extrude the surface mesh in the vertical to make a 3D mesh.

The most difficult aspect of extrusion is creating meshes that:
1. aren't huge numbers of cells
2. aren't huge cell thicknesses, especially near the surface
3. follow implied interfaces, e.g. bottom of soil and bottom of geologic layer

This is an iterative process that requires some care and some art.

## Extrude the 2D Mesh to make a 3D mesh

In [ ]:
# set the floor of the domain as max DTB
dtb_max = np.nanmax(m2.cell_data['dtb'].values)
m2.cell_data['dtb'] = m2.cell_data['dtb'].fillna(dtb_max)

total_thickness = np.ceil(dtb_max)
print(f"total thickness: {total_thickness} m")

#total_thickness = 50.

In [ ]:
# Generate a dz structure for the top 2m of soil
#
# here we try for 10 cells, starting at 5cm at the top and going to 50cm at the bottom of the 2m thick soil
dzs, res = watershed_workflow.mesh.optimizeDzs(0.05, 0.5, 2, 10)
print(dzs)
print(sum(dzs))

In [ ]:
# this looks like it would work out, with rounder numbers:
dzs_soil = [0.05, 0.05, 0.05, 0.12, 0.23, 0.5, 0.5, 0.5]
print(sum(dzs_soil))

In [ ]:
# total_thickness - minus 2m soil thickness, leaves us the rest to make up.
# optimize again...
dzs2, res2 = watershed_workflow.mesh.optimizeDzs(1, 10, total_thickness-2, 8)
#smallest geology layer around 1; largest geology layer around 10 m; total depth to fill = 38 m; try for about 8 layers

print(dzs2)
print(sum(dzs2))

dzs_geo = dzs2.astype(int)
print(dzs_geo)
print(sum(dzs_geo))

In [ ]:
if sum(dzs_geo) != total_thickness - 2:
    dzs_geo[-1] += total_thickness - 2 - sum(dzs_geo)
print(dzs_geo)
print(sum(dzs_geo))

In [ ]:
# layer extrusion
DTB = m2.cell_data['dtb'].values
soil_color = m2.cell_data['soil_color'].values
geo_color = m2.cell_data['geology_color'].values
soil_thickness = m2.cell_data['soil thickness'].values


# -- data structures needed for extrusion
layer_types = []
layer_data = []
layer_ncells = []
layer_mat_ids = []

# -- soil layer --
depth = 0
for dz in dzs_soil:
    depth += 0.5 * dz
    layer_types.append('constant')
    layer_data.append(dz)
    layer_ncells.append(1)
    
    # use glhymps params
    br_or_geo = np.where(depth < DTB, geo_color, 999)
    soil_or_br_or_geo = np.where(np.bitwise_and(soil_color > 0, depth < soil_thickness),
                                 soil_color,
                                 br_or_geo)

    layer_mat_ids.append(soil_or_br_or_geo)
    depth += 0.5 * dz
    
# -- geologic layer --
for dz in dzs_geo:
    depth += 0.5 * dz
    layer_types.append('constant')
    layer_data.append(dz)
    layer_ncells.append(1)
    
    geo_or_br = np.where(depth < DTB, geo_color, 999)

    layer_mat_ids.append(geo_or_br)
    depth += 0.5 * dz

# print the summary
watershed_workflow.mesh.Mesh3D.summarizeExtrusion(layer_types, layer_data, 
                                            layer_ncells, layer_mat_ids)

# downselect subsurface properties to only those that are used
layer_mat_id_used = list(np.unique(np.array(layer_mat_ids)))
subsurface_props_used = subsurface_props.loc[layer_mat_id_used]
subsurface_props_used


In [ ]:
subsurface_props_used.to_csv(f"./storecsv/{name}/{name}_subsurface_props_used__m{meshsize}_f{factor}_k{k_factor_text}.csv", index=False)

In [ ]:
len(subsurface_props_used)

In [ ]:
# extrude
m3 = watershed_workflow.mesh.Mesh3D.extruded_Mesh2D(m2, layer_types, layer_data, 
                                             layer_ncells, layer_mat_ids)

In [ ]:
print('2D labeled sets')
print('---------------')
for ls in m2.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

print('')
print('Extruded 3D labeled sets')
print('------------------------')
for ls in m3.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

print('')
print('Extruded 3D side sets')
print('---------------------')
for ls in m3.side_sets:
    print(f'{ls.setid} : FACE : {len(ls.cell_list)} : "{ls.name}"')
    


In [ ]:
with open(f'{savemesh_path}m3{choose_mesh_tag_name}_m{meshsize}_f{factor}_k{k_factor_text}.pkl', 'wb') as f:
    pickle.dump(m3, f)

In [ ]:
# save to disk
outputs['mesh_filename'] = f'../data-processed/{name}/{name}_wv2_m{meshsize}_f{factor}_k{k_factor_text}.exo'

try:
    os.remove(outputs['mesh_filename'])
except FileNotFoundError:
    pass
m3.writeExodus(outputs['mesh_filename'], 'material id')

## Collect the DayMet raster covering this area

Note that here we need two files -- the actual data and the typical year data.

The first cell downloads the raw data and generates the actual data file used by ATS, the second cell averages days, smooths the data, and writes a typical year.

In [ ]:
name

In [ ]:
generate_daymet = False

print(" A False value is advised here because this portion of daymet workflow is inherited from past notebook and is no longer functional with the newer (ww2) version")




In [ ]:
outputs['daymet_filename'] = f'./../../data-processed/{name}/{name}_DayMet_1980_2024.h5' # placing 1 layer up for the numerous testing

if generate_daymet:
    startdate = f"{start_year}-1-1"
    enddate = f"{end_year}-12-31"
    bounds = watershed.exterior().bounds
    
    source = watershed_workflow.sources.manager_daymet.FileManagerDaymet()
    data = source.get_data(bounds, crs, startdate, enddate)

    assert(len(data.collections) == 1)
    met_data = data.collections[0]
    met_data_ats = watershed_workflow.daymet.daymet_to_daily_averages(met_data)
    attrs = watershed_workflow.daymet.getAttributes(bounds, startdate, enddate)
    watershed_workflow.io.write_dataset_to_hdf5(outputs['daymet_filename'], met_data_ats.collections[0], attrs)

In [ ]:
outputs['daymet_spinup_filename'] = f'./../../data-processed/{name}/{name}_DayMet_typical_1980_2023.h5'

if generate_daymet:
    data_typ = source.get_data(bounds, crs, startdate, enddate)
    met_data_typ = data_typ.collections[0]
    
    logging.info("averaging daymet by taking the average for each day across the actual years.")
    window, poly_order = 61, 2
    for key in data.collections[0]:
        logging.info(f"smoothing {key} using savgol filter, window = {window} d, poly order = {poly_order}")
        met_data_typ.data[key] = watershed_workflow.utils.compute_average_year(met_data_typ.data[key], output_nyears=end_year-start_year+1, filter=True)    
    
    met_data_typ_ats = watershed_workflow.daymet.daymet_to_daily_averages(met_data_typ)
    watershed_workflow.io.write_dataset_to_hdf5(outputs['daymet_spinup_filename'], met_data_typ_ats.collections[0], attrs)

## Generate files for Leaf Area Index

In addition to meteorological forcing data, we need land cover leaf area index data.  Typically this can be smoother than the DayMet, and our default source for this, MODIS, is not currently automated.  Therefore this assumes you have already acquired the raw MODIS data (note -- get Pin to add his scripts for this!)

MODIS is available from 2002 to present -- we generate a typical year and can use that for all years, though probably we should do something smarter if possible.

In [ ]:
generate_modis = False

print(" A False value is advised here for generate_modis because this portion of modis workflow is inherited from past notebook and is no longer functional")


In [ ]:
def plot(df, form, axs):
    cm = watershed_workflow.colors.enumerated_colors(3)
    i = 0
    for k in df.keys():
        if not k.startswith('time'):
            axs[i].plot(df['time [d]'], df[k], form, color=cm[i])
            axs[i].set_title(k)
            i += 1

In [ ]:
# load the raw MODIS data and covert it to pandas
ats_raw_modis_filename = f'./../../data-processed/{name}/{name}_MODIS_LAI_07042002_04212025.h5'

if generate_modis:
    d = h5py.File(ats_raw_modis_filename,'r')
    df = pandas.DataFrame()
    for k in d.keys():
        df[k] = d[k][:]
    df['time [d]'] = df['time [s]']/86400
df if generate_modis else generate_modis

In [ ]:
if generate_modis:
    # interpolate this time series into a daily time series
    # ts = np.arange(8214, 14600, 1)
    ts = np.arange(df['time [d]'].values[-1]+1)
    df_interp = pandas.DataFrame()
    df_interp['time [d]'] = ts

    for k in df.keys():
        if k != 'time [s]':
            f = scipy.interpolate.interp1d(df['time [d]'][:], df[k][:])
            df_interp[k] = f(ts)

    df = df_interp
    df['datetime'] = pandas.to_datetime(df['time [d]'], unit='D', origin=pandas.Timestamp('2002-10-01'))
df if generate_modis else generate_modis

In [ ]:
if generate_modis:
    leap_index = []
    for i in range(len(df)):
        if '02-29' in str(df.iloc[i,-1]):
            leap_index.append(i)
            print(i, df.iloc[i,-1])

In [ ]:
if generate_modis:
    df.drop(leap_index, inplace=True)
    df.reset_index(drop=True, inplace=True)
    df['time [d]'] = np.arange(len(df))
df if generate_modis else generate_modis

In [ ]:
if generate_modis:
    for i in range(len(df)):
        if '02-29' in str(df.iloc[i,-1]):
            print(i, df.iloc[i,-1])

In [ ]:
if generate_modis:
    df.drop(columns=['datetime'], inplace=True)
df if generate_modis else generate_modis

In [ ]:
if generate_modis:
    # smooth the data
    df_smooth = pandas.DataFrame()
    df_smooth['time [d]'] = df['time [d]']
    for k in df.keys():
        if k != 'time [d]':
            df_smooth[k] = scipy.signal.savgol_filter(df[k], 101, 3)

    if generate_plots:
        # plot comparison
        fig = plt.figure()
        axs = fig.subplots(3,1)
        # plot(df, '-', axs)
        plot(df_smooth, '-', axs)
        plt.tight_layout()
        plt.show()
        
        
df_smooth if generate_modis else generate_modis

In [ ]:
# add time back and write to disk
outputs['modis_filename'] = f'./../../data-processed/{name}/{name}_MODIS_LAI_07042002_04212025_smoothed.h5'

if generate_modis:
    df_smooth['time [s]'] = df_smooth['time [d]']*86400
    with h5py.File(outputs['modis_filename'],'w') as fid:
        for k in df_smooth:
            fid.create_dataset(k, data=df_smooth[k][:])

In [ ]:
if generate_modis:
    df = df_smooth
    # split into n_years dataframes, one per year
    df_yr = []
    for year in range(20):
        yr = df.loc[df_interp['time [d]'] >= year*365]
        df_yr.append(yr.loc[yr['time [d]'] < (year+1)*365])

    # average across the years
    df_avg = pandas.DataFrame()
    for yr in df_yr:
        for k in yr.keys():
            if not k.startswith('time'):
                if k in df_avg:
                    df_avg[k] = df_avg[k].array + yr[k].array
                else:
                    df_avg[k] = yr[k].copy()

    for k in df_avg.keys():
        df_avg[k] = df_avg[k][:] / len(df_yr)

    df_avg['time [d]'] = df['time [d]']
    
df_avg if generate_modis else generate_modis

In [ ]:
if generate_modis:    
    if generate_plots:
        fig = plt.figure()
        axs = fig.subplots(3,1)
        plot(df_avg, '-', axs)
        plt.tight_layout()
        plt.show()

In [ ]:
nyears = 44#end_year-start_year+1 # to match DayMet 1980-2022
if generate_modis:
    # replicate nyears times to make nyears years (remem)
    # tile all data to repeat n_year times
    df_repeat = pandas.DataFrame()
    for key in df_avg:
        if not key.startswith('time'):
            df_repeat[key] = np.tile(df_avg[key].array, nyears)
            assert(len(df_repeat) == nyears*365)
    
    # time is simply daily data
    df_repeat['time [d]'] = np.arange(0., nyears * 365., 1.)
    df_repeat['time [s]'] = 86400*df_repeat['time [d]']
    
df_repeat if generate_modis else generate_modis

In [ ]:
if generate_modis and generate_plots:
    # plot this and make sure it looks right
    fig = plt.figure()
    axs = fig.subplots(3,1)
    plot(df_repeat, '-', axs)
    plt.tight_layout()
    plt.show()

In [ ]:
# write to disk
outputs['modis_typical_filename'] = f'./../../data-processed/{name}/{name}_MODIS_LAI_07042002_01012023_typical{nyears}yr.h5'

if generate_modis:
    with h5py.File(outputs['modis_typical_filename'],'w') as fid:
        for k in df_repeat:
            fid.create_dataset(k, data=df_repeat[k][:])

In [ ]:
# add 274 to typical
if generate_modis:
    d = h5py.File(outputs['modis_typical_filename'],'r')
    df = pandas.DataFrame()
    for k in d.keys():
        if k != 'time [d]':
            df[k] = d[k][:]
    d.close()
    df['time [s]'] += 274*86400
    with h5py.File(outputs['modis_typical_filename'],'w') as fid:
        for k in df:
            fid.create_dataset(k, data=df[k][:])
df if generate_modis else generate_modis

In [ ]:
# add 274 to smoothed
if generate_modis:
    d = h5py.File(outputs['modis_filename'],'r')
    df = pandas.DataFrame()
    for k in d.keys():
        if k != 'time [d]':
            df[k] = d[k][:]
    d.close()
    df['time [s]'] += (274+365*22)*86400
    with h5py.File(outputs['modis_filename'],'w') as fid:
        for k in df:
            fid.create_dataset(k, data=df[k][:])
df if generate_modis else generate_modis

## Write ATS input files

We now generate three input files -- two for spinup (steadystate solution and cyclic steadystate solution) and one for transient runs.

Steadystate has its own physics, but cyclic steadystate and transient share a common set of physics.  Each have their own met data strategy.

The first step is to generate the sections of xml that will replace parts of the template files.  This is done prior to loading any templates to make clear that these are totally generated from scratch using the ats_input_spec tool.

In [ ]:
# add the subsurface and surface domains
#
# Note this also adds a "computational domain" region to the region list, and a vis spec 
# for "domain"
def add_domains(main_list, mesh_filename, surface_region='surface', snow=True, canopy=True):
    ats_input_spec.public.add_domain(main_list, 
                                 domain_name='domain', 
                                 dimension=3, 
                                 mesh_type='read mesh file',
                                 mesh_args={'file':mesh_filename})
    if surface_region:
        main_list['mesh']['domain']['build columns from set'] = surface_region    
    
        # Note this also adds a "surface domain" region to the region list and a vis spec for 
        # "surface"
        ats_input_spec.public.add_domain(main_list,
                                domain_name='surface',
                                dimension=2,
                                mesh_type='surface',
                                mesh_args={'surface sideset name':'surface'})
    if snow:
        # Add the snow and canopy domains, which are aliases to the surface
        ats_input_spec.public.add_domain(main_list,
                                domain_name='snow',
                                dimension=2,
                                mesh_type='aliased',
                                mesh_args={'target':'surface'})
    if canopy:
        ats_input_spec.public.add_domain(main_list,
                                domain_name='canopy',
                                dimension=2,
                                mesh_type='aliased',
                                mesh_args={'target':'surface'})


In [ ]:
nlcd_indices

In [ ]:
nlcd_labels

In [ ]:
nlcd_labels_dict

In [ ]:
nlcd_indices = sorted(nlcd_labels_dict.keys())
nlcd_labels = [nlcd_labels_dict[i] for i in nlcd_indices]

print(nlcd_indices)
print(nlcd_labels)

In [ ]:

def add_land_cover(main_list):
    # next write a land-cover section for each NLCD type
    for nlcd_name in nlcd_labels:
        ats_input_spec.public.set_land_cover_default_constants(main_list, nlcd_name)

    land_cover_list = main_list["state"]["initial conditions"]["land cover types"]
    active_lc = set(np.unique(nlcd_color_new))
    # update some defaults
    # ['Other', 'Deciduous Forest', 'Evergreen Forest', 'Shrub/Scrub']
    # note, these are from the CLM Technical Note v4.5
    #
    # Rooting depth curves from CLM TN 4.5 table 8.3
    #
    # Note, the mafic potential values are likely pretty bad for the types of van Genuchten 
    # curves we are using (ETC -- add paper citation about this topic).  Likely they need
    # to be modified.  Note that these values are in [mm] from CLM TN 4.5 table 8.1, so the 
    # factor of 10 converts to [Pa]
    #
    # Note, albedo of canopy taken from CLM TN 4.5 table 3.1
    if 42 in active_lc:
        land_cover_list['Evergreen Forest']['rooting profile alpha [-]'] = 7.0
        land_cover_list['Evergreen Forest']['rooting profile beta [-]'] = 2.0
        land_cover_list['Evergreen Forest']['rooting depth max [m]'] = 10.0
        land_cover_list['Evergreen Forest']['capillary pressure at fully closed stomata [Pa]'] = 255000
        land_cover_list['Evergreen Forest']['capillary pressure at fully open stomata [Pa]'] = 66000 * .1
        land_cover_list['Evergreen Forest']['albedo of canopy [-]'] = 0.07

    if 41 in active_lc:
        land_cover_list['Deciduous Forest']['rooting profile alpha [-]'] = 6.0
        land_cover_list['Deciduous Forest']['rooting profile beta [-]'] = 2.0
        land_cover_list['Deciduous Forest']['rooting depth max [m]'] = 10.0
        land_cover_list['Deciduous Forest']['capillary pressure at fully closed stomata [Pa]'] = 224000
        land_cover_list['Deciduous Forest']['capillary pressure at fully open stomata [Pa]'] = 35000 * .10
        land_cover_list['Deciduous Forest']['albedo of canopy [-]'] = 0.1



In [ ]:
def add_domains(main_list, mesh_filename, surface_region="surface", snow=True, canopy=True):
    # 3D subsurface domain from Exodus mesh
    ats_input_spec.public.add_domain(
        main_list,
        domain_name="domain",
        dimension=3,
        mesh_type="read mesh file",
        mesh_args={"file": mesh_filename},
    )

    if surface_region:
        main_list["mesh"]["domain"]["build columns from set"] = surface_region

        # 2D surface mesh built from the surface sideset
        ats_input_spec.public.add_domain(
            main_list,
            domain_name="surface",
            dimension=2,
            mesh_type="surface",
            mesh_args={"surface sideset name": "surface"},
        )

    if snow:
        ats_input_spec.public.add_domain(
            main_list,
            domain_name="snow",
            dimension=2,
            mesh_type="aliased",
            mesh_args={"target": "surface"},
        )

    if canopy:
        ats_input_spec.public.add_domain(
            main_list,
            domain_name="canopy",
            dimension=2,
            mesh_type="aliased",
            mesh_args={"target": "surface"},
        )


def add_land_cover(main_list):
    # add one land-cover parameter block per active grouped NLCD class
    for nlcd_name in nlcd_labels:
        ats_input_spec.public.set_land_cover_default_constants(main_list, nlcd_name)

    land_cover_list = main_list["state"]["model parameters"]["land cover types"]
    active_lc = set(np.unique(nlcd_color_new))

    # CLM-based overrides for active classes

    # update some defaults
    # ['Other', 'Deciduous Forest', 'Evergreen Forest', 'Shrub/Scrub']
    # note, these are from the CLM Technical Note v4.5
    #
    # Rooting depth curves from CLM TN 4.5 table 8.3
    #
    # Note, the mafic potential values are likely pretty bad for the types of van Genuchten 
    # curves we are using (ETC -- add paper citation about this topic).  Likely they need
    # to be modified.  Note that these values are in [mm] from CLM TN 4.5 table 8.1, so the 
    # factor of 10 converts to [Pa]
    #
    # Note, albedo of canopy taken from CLM TN 4.5 table 3.1

    
    if 42 in active_lc:
        land_cover_list["Evergreen Forest"]["rooting profile alpha [-]"] = 7.0
        land_cover_list["Evergreen Forest"]["rooting profile beta [-]"] = 2.0
        land_cover_list["Evergreen Forest"]["rooting depth max [m]"] = 10.0
        land_cover_list["Evergreen Forest"]["capillary pressure at fully closed stomata [Pa]"] = 255000
        land_cover_list["Evergreen Forest"]["capillary pressure at fully open stomata [Pa]"] = 6600
        land_cover_list["Evergreen Forest"]["albedo of canopy [-]"] = 0.07

    if 41 in active_lc:
        land_cover_list["Deciduous Forest"]["rooting profile alpha [-]"] = 6.0
        land_cover_list["Deciduous Forest"]["rooting profile beta [-]"] = 2.0
        land_cover_list["Deciduous Forest"]["rooting depth max [m]"] = 10.0
        land_cover_list["Deciduous Forest"]["capillary pressure at fully closed stomata [Pa]"] = 224000
        land_cover_list["Deciduous Forest"]["capillary pressure at fully open stomata [Pa]"] = 3500
        land_cover_list["Deciduous Forest"]["albedo of canopy [-]"] = 0.1


"""
def soil_set_name(ats_id):
    if ats_id == 999:
        return "bedrock"
    source = subsurface_props_used.loc[ats_id]["source"]
    native_id = subsurface_props_used.loc[ats_id]["native_index"]
    if type(native_id) in [tuple, list]:
        native_id = native_id[0]
    return f"{source}-{native_id}"
"""

# add soil sets: note we need a way to name the set, so we use, e.g. SSURGO-MUKEY.
def soil_set_name(ats_id):
    if ats_id == 999:
        return 'bedrock'

    row = subsurface_props_used.loc[ats_id]
    source = row['source']

    if source == 'NRCS' and 'mukey' in row.index and pd.notna(row['mukey']):
        native_id = row['mukey']
    elif 'ID' in row.index and pd.notna(row['ID']):
        native_id = row['ID']
    elif 'name' in row.index and pd.notna(row['name']):
        native_id = row['name']
    else:
        native_id = ats_id

    if type(native_id) in [tuple, list]:
        native_id = native_id[0]

    return f"{source}-{native_id}"


def store_step_xml(main_list_copy, file_name):
    import os

    xml_path_trial = "./step_wise_xml"
    if not os.path.exists(xml_path_trial):
        os.makedirs(xml_path_trial)

    fullfilename = os.path.join(xml_path_trial, file_name)
    ats_input_spec.io.write(main_list_copy, fullfilename)
    main_xml_copy = ats_input_spec.io.to_xml(main_list_copy)
    print(fullfilename)


In [ ]:
# get an ATS "main" input spec list -- note, this is a dummy and is not used to write any files yet

# get the main primary list
main_list = ats_input_spec.public.get_main()
# store the main_list_xml step by step
store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'1_main_list_m{meshsize}_f{factor}_k{k_factor_text}.xml')

# add the lead pk
flow_pk = ats_input_spec.public.add_leaf_pk(
    main_list,
    'flow',
    main_list['cycle driver']['PK tree'],
    'pk-richards-flow-spec'
)
# store the updated xml
store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'2_add_leaf_pk_m{meshsize}_f{factor}_k{k_factor_text}.xml')

# add the mesh and all domains
mesh_filename = os.path.join('.', outputs['mesh_filename'])
# add domains
add_domains(main_list, mesh_filename)
# store the updated xml
store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'3_add_domains_m{meshsize}_f{factor}_k{k_factor_text}.xml')

# add labeled sets --> this includes adding the HUC, polygon and at observation points:::
for ls in m3.labeled_sets:
    ats_input_spec.public.add_region_labeled_set(main_list, ls.name, ls.setid, mesh_filename, ls.entity)
store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'4_add_HOPoly_labelSets_m{meshsize}_f{factor}_k{k_factor_text}.xml')

# add side sets --> This is part 2
for ss in m3.side_sets:
    ats_input_spec.public.add_region_labeled_set(main_list, ss.name, ss.setid, mesh_filename, 'FACE')
store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'5_add_HOPoly_SideSets_m{meshsize}_f{factor}_k{k_factor_text}.xml')

# add land cover
add_land_cover(main_list)
#print(main_list)
store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'6_add_landcover_m{meshsize}_f{factor}_k{k_factor_text}.xml')

c = 0
# add soil material ID regions, porosity, permeability, and WRMs
for ats_id in subsurface_props_used.index:
    props = subsurface_props_used.loc[ats_id]
    set_name = soil_set_name(ats_id)
    c = c + 1
    #print(f'Order number {c}')
    if props['van Genuchten n [-]'] < 1.5:
        smoothing_interval = 0.01
    else:
        smoothing_interval = 0.0

    ats_input_spec.public.add_soil_type(
        main_list,
        set_name,
        ats_id,
        mesh_filename,
        float(props['porosity [-]']),
        float(props['permeability [m^2]']), 1.e-7,
        float(props['van Genuchten alpha [Pa^-1]']),
        float(props['van Genuchten n [-]']),
        float(props['residual saturation [-]']),
        float(smoothing_interval)
    )
store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'7_add_soil_por_perm_wrm_m{meshsize}_f{factor}_k{k_factor_text}.xml')

# add domain water balance
ats_input_spec.public.add_observations_water_balance(
    main_list,
    "computational domain",
    "surface domain",
    "external sides"
)
store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'8_add_domain_waterBalance_m{meshsize}_f{factor}_k{k_factor_text}.xml')

# add watershed polygons water balance --- HUC
for region in watershed.df[names.NAME]:
    ats_input_spec.public.add_observations_water_balance(
        main_list,
        region,
        outlet_region=region + ' outlet'
    )
store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'9_add_HUC_waterBalance_m{meshsize}_f{factor}_k{k_factor_text}.xml')


# add watershed polygons water balance --- delineated polygons
if "polygon_labels" in locals() and polygon_labels:
    for poly_labels in polygon_labels:
        ats_input_spec.public.add_observations_water_balance(
            main_list,
            poly_labels,
            outlet_region=poly_labels + ' outlet'
        )
    store_step_xml(main_list_copy=copy.deepcopy(main_list), file_name=f'10_add_HUC_waterBalance_m{meshsize}_f{factor}_k{k_factor_text}.xml')

######## store the final xml ########
outputs['generated_ats'] = f'../data-processed/{name}/{name}_generated_ats_m{meshsize}_f{factor}_k{k_factor_text}.xml'
outputs['temp_gen_ats'] = f'./step_wise_xml/generated_ats_m{meshsize}_f{factor}_k{k_factor_text}.xml'
ats_input_spec.io.write(main_list, outputs['generated_ats'])
ats_input_spec.io.write(main_list, outputs['temp_gen_ats'])

# Turn this on if obscells are not required:
main_xml = ats_input_spec.io.to_xml(main_list)


In [ ]:
def populate_basic_properties(xml, main_xml, homogeneous_wrm=False, homogeneous_poro=False, homogeneous_perm=False):
    """This function updates an xml object with the above properties for mesh, regions, soil props, and lc props"""
    # find and replace the mesh list
    mesh_i = next(i for (i, el) in enumerate(xml) if el.get('name') == 'mesh')
    xml[mesh_i] = asearch.child_by_name(main_xml, 'mesh')

    # find and replace the regions list
    region_i = next(i for (i, el) in enumerate(xml) if el.get('name') == 'regions')
    xml[region_i] = asearch.child_by_name(main_xml, 'regions')

    fe_list = asearch.find_path(xml, ['state', 'evaluators'])

    # find and replace porosity, permeability
    if not homogeneous_poro:
        try:
            poro_i = next(i for (i, el) in enumerate(fe_list) if el.get('name') == 'base_porosity')
        except StopIteration:
            pass
        else:
            fe_list[poro_i] = asearch.find_path(main_xml, ['state', 'evaluators', 'base_porosity'])

    if not homogeneous_perm:
        try:
            perm_i = next(i for (i, el) in enumerate(fe_list) if el.get('name') == 'permeability')
        except StopIteration:
            pass
        else:
            fe_list[perm_i] = asearch.find_path(main_xml, ['state', 'evaluators', 'permeability'])

    # find and replace the WRMs list
    if not homogeneous_wrm:
        main_state = asearch.child_by_name(main_xml, 'state')
        xml_state = asearch.child_by_name(xml, 'state')

        main_model_params = asearch.child_by_name(main_state, 'model parameters')
        xml_model_params = asearch.child_by_name(xml_state, 'model parameters')

        wrm_i = next(i for (i, el) in enumerate(xml_model_params) if el.get('name') == 'WRM parameters')
        xml_model_params[wrm_i] = asearch.child_by_name(main_model_params, 'WRM parameters')

    # find and replace land cover
    try:
        main_state = asearch.child_by_name(main_xml, 'state')
        xml_state = asearch.child_by_name(xml, 'state')

        main_model_params = asearch.child_by_name(main_state, 'model parameters')
        xml_model_params = asearch.child_by_name(xml_state, 'model parameters')

        lc_i = next(i for (i, el) in enumerate(xml_model_params) if el.get('name') == 'land cover types')
    except (StopIteration, aerrors.MissingXMLError):
        pass
    else:
        xml_model_params[lc_i] = asearch.child_by_name(main_model_params, 'land cover types')


For the first file, we load a spinup template and write the needed quantities into that file, saving it to the appropriate run directory.  Note there is no DayMet or land cover or LAI properties needed for this run.  The only property that is needed is the domain-averaged, mean annual rainfall rate.  We then take off some for ET (note too wet spins up faster than too dry, so don't take off too much...).

In [ ]:
if generate_daymet:
    # calculate the basin-averaged, annual-averaged precip rate
    precip_total = ats_typ['precipitation rain [m s^-1]'] + ats_typ['precipitation snow [m SWE s^-1]']
    mean_precip = precip_total.mean()
else:
    rain = 'precipitation rain [m s^-1]'
    snow = 'precipitation snow [m SWE s^-1]'
    try:
        with h5py.File(outputs['daymet_spinup_filename'], 'r') as fid:
            mean_precip = np.array([fid[rain][ts][:].mean() + fid[snow][ts][:].mean() for ts in fid[rain].keys()]).mean()
    except:
        mean_precip = 5e-8

# or forced setup
mean_precip = 5e-8
logging.info(f'Mean annual precip rate [m s^-1] = {mean_precip}')

In [ ]:
def write_spinup_steadystate(name, mean_precip, **kwargs):
    name = create_unique_name(name, **kwargs)
    logging.info(f'Writing spinup steady-state: {name}')
    
    # write the spinup xml file
    # load the template file
    xml = aio.fromFile('steadystate-template.xml')

    # populate basic properties for mesh, regions, and soil properties
    populate_basic_properties(xml, main_xml, **kwargs)

    # set the mean avg source as 60% of mean precip
    precip_el = asearch.find_path(
        xml,
        ['state', 'evaluators', 'surface-precipitation', 'function-constant', 'value']
    )
    precip_el.setValue(mean_precip * .7)

    # write to disk
    outputs[f'spinup_steadystate_{name}_filename'] = f'../run0-spinup_steadystate/run0_{name}_m{meshsize}_f{factor}_k{k_factor_text}.xml'

    try:
        os.mkdir('../run0-spinup_steadystate')
    except FileExistsError:
        pass

    aio.toFile(xml, outputs[f'spinup_steadystate_{name}_filename'])

    # make a run directory
    outputs[f'spinup_steadystate_{name}_rundir'] = f'../run0-spinup_steadystate/{name}_m{meshsize}_f{factor}_k{k_factor_text}'

    try:
        os.mkdir(outputs[f'spinup_steadystate_{name}_rundir'])
    except FileExistsError:
        pass


In [ ]:
f'../run0-spinup_steadystate/{name}-0'

In [ ]:
outputs

For the second file, we load a transient run template.  This file needs the basics, plus DayMet and LAI as the "typical year data".  Also we set the run directory that will be used for the steadystate run.

For the third file, we load a transient run template as well.  This file needs the basics, DayMet with the actual data, and we choose for this run to use the MODIS typical year.  MODIS is only available for 2002 on, so if we didn't need 1980-2002 we could use the real data, but for this run we want a longer record.

In [ ]:
def write_transient(name, cyclic_steadystate=False, start_year=1980, end_year=2025, **kwargs):
    # make a unique name based on options
    name = create_unique_name(name, **kwargs)
    logging.info(f'Writing transient: {name}')

    if cyclic_steadystate:
        prefix = 'spinup_cyclic'
        # start_year = 1980
        # end_year = 2022
        previous = 'spinup_steadystate'
        runnum = 'run1'
        template_filename = 'cyclic_steadystate-template.xml'
    else:
        prefix = 'transient'
        previous = 'spinup_cyclic'
        runnum = 'run2'
        template_filename = 'transient-template.xml'
    
    # write the cyclic spinup xml file
    # load the template file
    xml = aio.fromFile(template_filename)

    # populate basic properties for mesh, regions, and soil properties
    populate_basic_properties(xml, main_xml, **kwargs)

    # update the DayMet filenames
    if cyclic_steadystate:
        daymet_filename = outputs['daymet_spinup_filename']
    else:
        daymet_filename = outputs['daymet_filename']

    for var in ['surface-incoming_shortwave_radiation',
                'surface-precipitation_rain',
                'snow-precipitation',
                'surface-air_temperature',
                'surface-vapor_pressure_air',
                'surface-temperature',
                'canopy-temperature']:
        try:
            par = asearch.find_path(xml, ['state', 'evaluators', var, 'file'])
        except aerrors.MissingXMLError:
            pass
        else:
            par.setValue(os.path.join('.', daymet_filename))

    # update the LAI filenames
    for par in asearch.findall_path(xml, ['canopy-leaf_area_index', 'file']):
        if cyclic_steadystate:
            par.setValue(os.path.join('.', outputs['modis_typical_filename']))
        else:
            par.setValue(os.path.join('.', outputs['modis_filename']))
    
    # update the start and end time -- start at Oct 1 of year 0, end 10 years later
    start_day = 274 + 365 * (start_year - 1980)
    par = asearch.find_path(xml, ['cycle driver', 'start time'])
    par.setValue(start_day)

    end_day = 274 + 365 * (end_year - 1980)
    par = asearch.find_path(xml, ['cycle driver', 'end time'])
    par.setValue(end_day)
    
    # update the restart filenames
    for var in asearch.findall_path(xml, ['initial condition', 'restart file']):
        var.setValue(os.path.join('.', outputs[f'{previous}_{name}_rundir'], 'checkpoint_final.h5'))

    # update the observations list
    obs = next(i for (i, el) in enumerate(xml) if el.get('name') == 'observations')
    xml[obs] = asearch.child_by_name(main_xml, 'observations')
   
    # write to disk and make a directory for running the run
    outputs[f'{prefix}_{name}_filename'] = f'../{runnum}-{prefix}/{runnum}_{name}_m{meshsize}_f{factor}_k{k_factor_text}.xml'

    filename = outputs[f'{prefix}_{name}_filename']
    outputs[f'{prefix}_{name}_rundir'] = f'../{runnum}-{prefix}/{name}_m{meshsize}_f{factor}_k{k_factor_text}'
    rundir = outputs[f'{prefix}_{name}_rundir']
    
    try:
        os.mkdir(f'../{runnum}-{prefix}')
    except FileExistsError:
        pass

    aio.toFile(xml, filename)
    try:
        os.mkdir(rundir)
    except FileExistsError:
        pass


def create_unique_name(name, homogeneous_wrm=False, homogeneous_poro=False, homogeneous_perm=False):
    suffix = '_h'
    if homogeneous_perm:
        suffix += 'K'
    if homogeneous_poro:
        suffix += 'p'
    if homogeneous_wrm:
        suffix += 'w'
    if suffix == '_h':
        suffix = ''
    return name + suffix




In [ ]:
outputs

In [ ]:
# create the fully-heterogeneous runs
if include_heterogeneous:
    write_spinup_steadystate(name, mean_precip)
    write_transient(name, True)
    write_transient(name, False)

# create homogeneous runs
if include_homogeneous:
    write_spinup_steadystate(name, mean_precip, homogeneous_wrm=True, homogeneous_poro=True, homogeneous_perm=True)
    write_transient(name, True, homogeneous_wrm=True, homogeneous_poro=True, homogeneous_perm=True)
    write_transient(name, False, homogeneous_wrm=True, homogeneous_poro=True, homogeneous_perm=True)
    
if include_homogeneous_wrm:
    write_spinup_steadystate(name, mean_precip, homogeneous_wrm=True, homogeneous_poro=False, homogeneous_perm=False)
    write_transient(name, True, homogeneous_wrm=True, homogeneous_poro=False, homogeneous_perm=False)
    write_transient(name, False, homogeneous_wrm=True, homogeneous_poro=False, homogeneous_perm=False)
    
if include_homogeneous_wrm_porosity:
    write_spinup_steadystate(name, mean_precip, homogeneous_wrm=True, homogeneous_poro=True, homogeneous_perm=False)
    write_transient(name, True, homogeneous_wrm=True, homogeneous_poro=True, homogeneous_perm=False)
    write_transient(name, False, homogeneous_wrm=True, homogeneous_poro=True, homogeneous_perm=False)
    
if include_homogeneous_wrm_permeability:
    write_spinup_steadystate(name, mean_precip, homogeneous_wrm=True, homogeneous_poro=False, homogeneous_perm=True)
    write_transient(name, True, homogeneous_wrm=True, homogeneous_poro=False, homogeneous_perm=True)
    write_transient(name, False, homogeneous_wrm=True, homogeneous_poro=False, homogeneous_perm=True)


In [ ]:
name

In [ ]:
outputs

In [ ]:
try:
    newfile0, newfile1, newfile2
except NameError:
    newfile0, newfile1, newfile2 = (
        outputs[f'spinup_steadystate_{name}_filename'],
        outputs[f'spinup_cyclic_{name}_filename'],
        outputs[f'transient_{name}_filename'],
    )


In [ ]:
print(newfile0)
print(newfile1)
print(newfile2)

In [ ]:
outputs

#### Remove or reduce the significant digits over all the xmls run0, run1 and run2 (may not need this)

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

def truncate_double_values(input_xml, tag_name=None, sig_digits=10):
    input_path = Path(input_xml)

    if tag_name is None:
        output_path = input_path
    else:
        output_path = input_path.with_name(f"{input_path.stem}_{tag_name}.xml")

    tree = ET.parse(input_path)
    root = tree.getroot()

    for param in root.iter('Parameter'):
        if param.get('type') == 'double':
            val_str = param.get('value')
            if val_str is None:
                continue
            try:
                val_float = float(val_str)
                val_short = f"{val_float:.{sig_digits}g}"
                if val_str != val_short:
                    param.set('value', val_short)
            except ValueError:
                continue

    tree.write(output_path, encoding='UTF-8', xml_declaration=False)
    if tag_name is None:
        print(f"XML overwritten: {output_path}")
    else:
        print(f"Cleaned XML written to: {output_path}")
    return output_path


In [ ]:
tag_name = None
newfile0 = truncate_double_values(newfile0, tag_name=tag_name, sig_digits=8)
newfile1 = truncate_double_values(newfile1, tag_name=tag_name, sig_digits=8)
newfile2 = truncate_double_values(newfile2, tag_name=tag_name, sig_digits=8)


In [ ]:
print(newfile0)
print(newfile1)
print(newfile2)

#### Add the observations csv "surface-water-table depth" in one of the HUCs for run1 and run2 --> Apparently having this in ATS observations will also lead to generating/adding the water table depth in the visualizaiton files 

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

def serialize_with_newlines(elem):
    rough_string = ET.tostring(elem, encoding='unicode')
    return rough_string.replace('><', '>\n<')

def insert_swtd_trigger_minimal(input_xml, tag_name=None):
    input_path = Path(input_xml)
    output_path = input_path if tag_name is None else input_path.with_name(f"{input_path.stem}_{tag_name}.xml")

    tree = ET.parse(input_path)
    root = tree.getroot()

    # Locate <ParameterList name="observations">
    obs_block = root.find(".//ParameterList[@name='observations']")
    if obs_block is None:
        raise ValueError("No <ParameterList name='observations'> block found.")

    # Gather existing water-balance blocks
    wb_blocks = [
        pl.get("name") for pl in obs_block.findall("ParameterList")
        if pl.get("name", "").startswith("water_balance")
    ]
    if not wb_blocks:
        raise ValueError("No water_balance_* blocks found.")

    if "water_balance_computational_domain" in wb_blocks:
        wb_blocks.remove("water_balance_computational_domain")
        wb_blocks.append("water_balance_computational_domain")

    src_name = wb_blocks[0]

    # Determine region core
    if src_name == "water_balance_computational_domain":
        region_core = "surface domain"
    else:
        region_core = src_name.replace("water_balance_", "")

    # Build the trigger block exactly as specified
    trigger = ET.Element("ParameterList", name="swtd_trigger", type="ParameterList")

    ET.SubElement(trigger, "Parameter", name="observation output filename", type="string", value="swtd_trigger.csv")
    ET.SubElement(trigger, "Parameter", name="delimiter", type="string", value=",")
    ET.SubElement(trigger, "Parameter", name="time units", type="string", value="d")

    observed_q = ET.SubElement(trigger, "ParameterList", name="observed quantities", type="ParameterList")
    swtd = ET.SubElement(observed_q, "ParameterList", name="water table depth [m]", type="ParameterList")
    ET.SubElement(swtd, "Parameter", name="variable", type="string", value="surface-water_table_depth")
    ET.SubElement(swtd, "Parameter", name="region", type="string", value=f"{region_core} surface")
    ET.SubElement(swtd, "Parameter", name="location name", type="string", value="cell")
    ET.SubElement(swtd, "Parameter", name="functional", type="string", value="average")
    ET.SubElement(swtd, "Parameter", name="time integrated", type="bool", value="false")

    ET.SubElement(trigger, "Parameter", name="times start period stop", type="Array(double)", value="{0.0, 1.0, -1.0}")
    ET.SubElement(trigger, "Parameter", name="times start period stop units", type="string", value="d")
    # Append to observation block
    obs_block.append(trigger)

    from xml.dom import minidom
    rough_string = ET.tostring(root, encoding="unicode")
    pretty_string = minidom.parseString(rough_string).toprettyxml(indent="  ")
    pretty_lines = [line for line in pretty_string.splitlines() if line.strip() and not line.startswith("<?xml")]
    cleaned_string = "\n".join(pretty_lines)
    with open(output_path, "w") as f:
        f.write(cleaned_string)
    msg = "overwritten" if tag_name is None else "written"
    print(f"swtd_trigger {msg} → {output_path}")

    return output_path

In [ ]:
tag_name = None #'swtd'
newfile1 = insert_swtd_trigger_minimal(newfile1, tag_name=tag_name)
newfile2 = insert_swtd_trigger_minimal(newfile2,tag_name=tag_name)

In [ ]:
print(newfile0)
print(newfile1)
print(newfile2)

In [ ]:
logging.info('this workflow is a total success')